# Notebook 3 — NumPy & Pandas : niveau AVANCÉ

**Objectif** : tester une véritable maîtrise. Ici, **personne ne vous dit quelle méthode utiliser**. Vous devez analyser le problème, choisir les bons outils, structurer votre code et **valider** vos résultats.

**Pré-requis** : `01_DEBUTANT.ipynb` et `02_INTERMEDIAIRE.ipynb` acquis.

**Règles du jeu**
- Les consignes décrivent *quoi* produire et *comment vérifier*, rarement *comment faire*.
- Interdiction (sauf mention contraire) des boucles Python sur les lignes des gros jeux de données : vectorisez.
- Validez toujours : comparez à une méthode lente mais évidente sur un petit échantillon (`np.allclose`, `assert`).
- Pensez à la **mémoire** et au **temps** : sur des centaines de milliers de lignes, les erreurs d'approche se paient.
- Si vous bloquez, relisez le chapitre du cours correspondant avant de regarder la correction.

**Contenu** : 16 problèmes complexes (NumPy multi-dimensionnel, broadcasting, vectorisation, gros volumes, nettoyage de données imbriquées, jointures temporelles et par intervalle, séries temporelles en panel, statistiques appliquées, pipelines) puis **2 mini-projets d'analyse complets**.

**Niveau** : 🔴 avancé pour tous les exercices.

> Exécutez d'abord les cellules de préparation. Les jeux de volume moyen prennent quelques secondes à générer.

## Sommaire des exercices

- Exercice 1 — Standardisation d'un lot d'images (broadcasting 4D) *(🔴 avancé)*
- Exercice 2 — Plus proches voisins sur 20 000 vecteurs (bloc par bloc) *(🔴 avancé)*
- Exercice 3 — Impôt progressif sur un million de revenus *(🔴 avancé)*
- Exercice 4 — Simulation Monte-Carlo de rupture de stock *(🔴 avancé)*
- Exercice 5 — Lecture par blocs : des agrégats exacts sans tout charger *(🔴 avancé)*
- Exercice 6 — Agrégations multiples et indicateurs personnalisés *(🔴 avancé)*
- Exercice 7 — Dernier achat par client : quelle méthode choisir ? *(🔴 avancé)*
- Exercice 8 — Jointure par intervalle, jointure temporelle et anti-jointure *(🔴 avancé)*
- Exercice 9 — Optimiser une analyse lente (profilage) *(🔴 avancé)*
- Exercice 10 — Aplatir et nettoyer un JSON imbriqué *(🔴 avancé)*
- Exercice 11 — Du format large au format long, et retour *(🔴 avancé)*
- Exercice 12 — Panel de séries temporelles : tendances, alertes et autocorrélation *(🔴 avancé)*
- Exercice 13 — Le paradoxe de Simpson (calculs de calcul rénal) *(🔴 avancé)*
- Exercice 14 — Test A/B : p-valeur, puissance et permutation vectorisée *(🔴 avancé)*
- Exercice 15 — Construire un pipeline de nettoyage testable et traçable *(🔴 avancé)*
- Exercice 16 — ACP à la main : valeurs propres, SVD et reconstruction *(🔴 avancé)*
- Exercice 17 — Fenêtres glissantes vectorisées : corrélation, drawdown et recherche de motif *(🔴 avancé)*
- Exercice 18 — PROJET 1 — Moto-taxis de Cotonou : analyse complète *(🏁 mini-projet)*
- Exercice 19 — PROJET 2 — Rendements agricoles : analyse complète *(🏁 mini-projet)*
- **CORRECTIONS ET EXPLICATIONS** (en fin de notebook — ne regardez qu'après avoir cherché !)

In [ ]:
import time, os, tempfile
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 150)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

def chrono(f, *a, **k):
    t = time.perf_counter(); r = f(*a, **k); return r, time.perf_counter() - t

---
# PARTIE I — NumPy multi-dimensionnel, broadcasting et vectorisation

### Jeux de données de cette partie

In [ ]:
# Lot de 64 images couleur 24x24 (n_images, canaux, hauteur, largeur), avec des luminosités différentes par image
rng_i = np.random.default_rng(100)
images = (rng_i.gamma(2.0, 30.0, size=(64, 3, 24, 24)) * np.array([1.0, 0.8, 1.3])[None, :, None, None]
          + rng_i.normal(0, 25, (64, 1, 1, 1)))

# 3 000 documents décrits par des vecteurs de 32 dimensions, répartis en 20 thèmes (clusters)
rng_e = np.random.default_rng(101)
themes = rng_e.integers(0, 20, 3000)
centres_t = rng_e.normal(size=(20, 32)) * 3
embeddings = centres_t[themes] + rng_e.normal(size=(3000, 32))
embeddings_xl = (centres_t[rng_e.integers(0, 20, 20000)] + rng_e.normal(size=(20000, 32))).astype(np.float32)

# 1 000 000 de revenus annuels (FCFA) pour l'impôt progressif
rng_p = np.random.default_rng(102)
revenus = np.round(rng_p.lognormal(np.log(1_800_000), 0.8, 1_000_000), -2)
print(images.shape, embeddings.shape, embeddings_xl.shape, revenus.shape)

### Exercice 1 — Standardisation d'un lot d'images (broadcasting 4D)
*Niveau : 🔴 avancé*

`images` est un tableau `(64, 3, 24, 24)` : 64 images, 3 canaux, 24×24 pixels.

1. Standardisez **chaque canal de chaque image séparément** (moyenne 0, écart-type 1 sur les pixels). Vérifiez que les moyennes et écarts-types par (image, canal) valent 0 et 1.
2. Standardisez cette fois **par canal sur l'ensemble du lot** (une moyenne et un écart-type par canal, communs aux 64 images). Les moyennes par (image, canal) sont-elles maintenant nulles ? Pourquoi ?
3. Étirez le contraste de chaque image (min-max sur tous ses canaux et pixels) pour que les valeurs soient dans [0, 1].
4. Appliquez à tous les pixels de toutes les images une transformation de couleur linéaire `M` (3×3) — `M = [[0.3,0.6,0.1],[0.5,0.4,0.1],[0.2,0.2,0.6]]` — en **une seule instruction** (`einsum`), puis vérifiez le résultat sur un pixel en calculant à la main `M @ pixel`.
5. Aplatissez le lot en `(64, 1728)` ; ce tableau est-il une **vue** ou une copie ? (`np.shares_memory`)

### Exercice 2 — Plus proches voisins sur 20 000 vecteurs (bloc par bloc)
*Niveau : 🔴 avancé*

`embeddings` : 3 000 vecteurs de dimension 32, répartis en 20 thèmes (`themes` donne l'étiquette réelle). `embeddings_xl` : 20 000 vecteurs `float32`.

1. Normalisez chaque vecteur de `embeddings` (norme 1) puis calculez la matrice de **similarité cosinus** `(3000, 3000)` par un produit matriciel.
2. Pour chaque document, trouvez ses **3 voisins les plus proches** (hors lui-même) **sans trier toute la matrice** : utilisez `np.argpartition`. Vérifiez sur 100 documents que le résultat coïncide (en tant qu'ensemble) avec un tri complet.
3. Quel pourcentage de voisins proches partage le même thème que le document ? (cela mesure la qualité de l'espace d'embedding)
4. Pour `embeddings_xl`, la matrice complète `(20000, 20000)` en float32 pèserait 1,6 Go : calculez la même chose **par blocs de 500 lignes** pour ne jamais dépasser ~40 Mo de matrice temporaire. Comparez le temps et vérifiez que le plus proche voisin est identique à celui obtenu en calculant un bloc de contrôle en entier.

### Exercice 3 — Impôt progressif sur un million de revenus
*Niveau : 🔴 avancé*

Barème annuel (FCFA) : 0 % jusqu'à 800 000 ; 10 % de 800 000 à 2 000 000 ; 15 % de 2 000 000 à 4 000 000 ; 20 % de 4 000 000 à 8 000 000 ; 30 % au-delà de 8 000 000. (Impôt **marginal par tranche** : chaque taux ne s'applique qu'à la part du revenu située dans la tranche.)

1. Écrivez une fonction `impot_boucle(r)` pour **un** revenu avec une boucle sur les tranches (testez-la à la main : un revenu de 3 000 000 doit donner 120 000 + 0,15 × 1 000 000 = 270 000).
2. Écrivez `impot_broadcast(revenus)` qui traite les **1 000 000** de revenus par broadcasting `(n, 1)` contre `(5,)` avec `np.clip`.
3. Écrivez `impot_searchsorted(revenus)` qui utilise `np.searchsorted` et un tableau précalculé d'impôts cumulés aux bornes (sans tableau `(n, 5)`).
4. Vérifiez que les trois approches concordent sur un échantillon de 1 000 revenus, puis comparez temps d'exécution et mémoire intermédiaire pour la totalité.
5. Calculez, par décile de revenu, le revenu moyen, l'impôt moyen et le **taux effectif** (impôt/revenu). Le système est-il progressif (le taux effectif croît-il ?) ?
6. Quelle part de l'impôt total est payée par les 10 % les plus riches ?

### Exercice 4 — Simulation Monte-Carlo de rupture de stock
*Niveau : 🔴 avancé*

Un commerce vend un produit pendant 90 jours. La demande quotidienne suit une loi de Poisson dont la moyenne est `λ = 18` en semaine et `λ = 30` le week-end, **multipliée chaque jour par un aléa** `exp(N(0, 0.25²))` (jours « chauds »/« froids »). Le stock initial est `S0 = 300`. Tous les **7 jours** (jours 7, 14, …, 84), un réapprovisionnement de `Q` unités arrive **avant** l'ouverture. Les ventes manquées en cas de rupture sont perdues (pas de stock négatif).

1. Écrivez une fonction `simuler(Q, n_scen=100_000, seed=0)` qui simule `n_scen` scénarios **simultanément** : une boucle sur les 90 jours est autorisée, mais **aucune boucle sur les scénarios**. Elle retourne la matrice des ventes perdues `(n_scen, 90)` et la matrice des stocks de fin de journée.
2. Estimez, pour `Q = 150`, la probabilité qu'une rupture se produise au moins une fois sur 90 jours, et le nombre moyen de ventes perdues.
3. Évaluez de façon vectorisée une grille `Q ∈ {120, 130, …, 200}` : probabilité de rupture, ventes perdues moyennes, stock moyen. Astuce : ajoutez une dimension `Q` aux tableaux.
4. Coût total = `15 × ventes perdues + 0,05 × stock moyen quotidien × 90 + 40 × nombre de livraisons` (en milliers de FCFA). Quelle valeur de `Q` minimise le coût ?
5. Fournissez un intervalle de confiance à 95 % de la probabilité de rupture pour la valeur de `Q` optimale.

---
# PARTIE II — Gros volumes et performance

### Jeu de données : 800 000 transactions de magasins
Écrites dans un fichier CSV temporaire (comme un export réel), pour pratiquer la lecture par blocs.

In [ ]:
rng_t = np.random.default_rng(300)
N = 800_000
magasins = np.array([f"MAG-{i:02d}" for i in range(50)])
produits_ids = np.array([f"P{i:03d}" for i in range(200)])
poids_mag = rng_t.dirichlet(np.ones(50) * 3)
poids_prod = rng_t.dirichlet(np.ones(200) * 0.8)
prix_prod = np.round(rng_t.lognormal(np.log(3000), 0.9, 200), -1)
ip = rng_t.choice(200, N, p=poids_prod)
qte = rng_t.choice([1, 2, 3, 4, 5, 10], N, p=[0.55, 0.2, 0.1, 0.07, 0.05, 0.03])
ts = pd.Timestamp("2024-01-01") + pd.to_timedelta(rng_t.integers(0, 366 * 24 * 3600, N), unit="s")
tx_df = pd.DataFrame({
    "id_tx": np.arange(N), "ts": ts, "magasin": magasins[rng_t.choice(50, N, p=poids_mag)],
    "client": rng_t.integers(1, 120_001, N), "produit": produits_ids[ip], "quantite": qte,
    "montant": np.round(prix_prod[ip] * qte * rng_t.uniform(0.9, 1.0, N), 0),
}).sort_values("ts").reset_index(drop=True)
dossier_gros = tempfile.mkdtemp()
chemin_tx = os.path.join(dossier_gros, "transactions.csv")
tx_df.to_csv(chemin_tx, index=False)
print(tx_df.shape, f"{os.path.getsize(chemin_tx) / 1e6:.0f} Mo sur disque", f"{tx_df.memory_usage(deep=True).sum() / 1e6:.0f} Mo en mémoire")

### Exercice 5 — Lecture par blocs : des agrégats exacts sans tout charger
*Niveau : 🔴 avancé*

On suppose que le fichier `chemin_tx` est **trop gros pour tenir en mémoire**. Sans jamais charger plus de 100 000 lignes à la fois (`chunksize`), calculez :

1. Le chiffre d'affaires total, le nombre de transactions et le **CA par magasin** (agrégation incrémentale : combinez les résultats partiels de chaque bloc).
2. Le nombre de **clients distincts** (exact), via un ensemble ou un tableau d'identifiants distincts mis à jour bloc par bloc.
3. La **médiane approchée** du montant, par un histogramme à bornes fixes (par exemple 2 000 classes entre 0 et le max connu) accumulé bloc par bloc. Comparez à la médiane exacte (obtenue en chargeant tout, pour vérification seulement). Pourquoi la médiane exacte est-elle impossible à obtenir par simple combinaison de médianes partielles ?
4. Reprenez la lecture en n'ouvrant que les colonnes utiles, avec des types économes (`category`, `float32`, `int32`). Quelle est l'économie mémoire sur le fichier complet ?
5. Vérifiez que tous vos résultats exacts correspondent à ceux de `tx_df`.

### Exercice 6 — Agrégations multiples et indicateurs personnalisés
*Niveau : 🔴 avancé*

Avec `tx_df` (en mémoire), produisez **en un seul tableau** `bilan` (un magasin par ligne) :

- `ca`, `nb_tx`, `panier_moyen` (CA / nombre de transactions) ;
- `prix_moyen_unit` : prix unitaire moyen **pondéré par la quantité** (Σ montant / Σ quantité) ;
- `p90_montant` : 90ᵉ percentile des montants ;
- `nb_clients` : clients distincts ;
- `jours_actifs` : nombre de jours avec au moins une transaction ;
- `produit_top` : le produit le plus vendu **en quantité** du magasin ;
- `hhi` : indice de concentration de Herfindahl-Hirschman sur le CA par produit (Σ des parts² ; 1/200 = 0,005 si parfaitement réparti, 1 si un seul produit) ;
- `part_top5` : part du CA réalisée par les 5 meilleurs produits du magasin.

Contraintes : pas de `groupby.apply` ligne à ligne sur 800 000 lignes ; privilégiez des groupby successifs puis `join`. Vérifiez deux magasins à la main (boucle) pour valider. Quels magasins ont la plus forte concentration ?

### Exercice 7 — Dernier achat par client : quelle méthode choisir ?
*Niveau : 🔴 avancé*

Pour chaque client de `tx_df`, on veut la **ligne de sa dernière transaction** (par `ts`).

1. Implémentez au moins **quatre** méthodes différentes (par exemple : `sort_values` + `drop_duplicates(keep="last")`, `groupby().idxmax()`, `groupby().tail(1)`, `np.lexsort` + masque de dernier élément de chaque groupe).
2. Vérifiez qu'elles retournent **exactement** les mêmes lignes (même ensemble d'`id_tx`).
3. Mesurez et classez leurs temps d'exécution.
4. Même question pour « le **premier** achat » et « le **plus gros** achat » de chaque client : lesquelles des méthodes se généralisent ?
5. Sans tout recharger, calculez pour chaque client le **nombre de jours** entre son premier et son dernier achat, ainsi que le délai médian entre deux achats consécutifs (indice : `groupby().diff()`).

### Exercice 8 — Jointure par intervalle, jointure temporelle et anti-jointure
*Niveau : 🔴 avancé*

```python
rng_j = np.random.default_rng(400)
dates = pd.date_range("2024-01-01", "2024-12-31", freq="D")
# taux de change XOF pour 1 USD : publié uniquement les jours ouvrés
taux = pd.DataFrame({"date": dates[dates.dayofweek < 5]})
taux["xof_par_usd"] = (600 + np.cumsum(rng_j.normal(0, 1.2, len(taux)))).round(2)
# 5 000 paiements en USD avec horodatage précis
paiements = pd.DataFrame({"ts": pd.Timestamp("2024-01-01") + pd.to_timedelta(np.sort(rng_j.integers(0, 365 * 86400, 5000)), unit="s"),
                          "usd": rng_j.uniform(10, 500, 5000).round(2)})
# promotions : (produit, début, fin, remise) – certaines se chevauchent
promos = pd.DataFrame({"produit": ["P001", "P001", "P002", "P003", "P003"],
                       "debut": pd.to_datetime(["2024-02-01", "2024-02-15", "2024-03-01", "2024-01-10", "2024-06-01"]),
                       "fin":   pd.to_datetime(["2024-02-29", "2024-03-15", "2024-03-31", "2024-02-10", "2024-06-30"]),
                       "remise": [0.10, 0.20, 0.15, 0.05, 0.25]})
```
1. **Jointure temporelle** : associez à chaque paiement le dernier taux publié **avant** son horodatage (`merge_asof`), avec une tolérance de 4 jours. Calculez `xof`. Vérifiez que les paiements du samedi/dimanche utilisent le taux du vendredi.
2. **Jointure par intervalle** : pour les transactions de `tx_df` portant sur `P001`, `P002`, `P003`, déterminez la remise applicable à la date de la transaction (jointure sur `produit` puis filtre `debut ≤ date ≤ fin`). Quand deux promotions se chevauchent, retenez la **plus forte remise**. Les transactions hors promotion ont une remise de 0. Contrôlez que le nombre de lignes final est égal au nombre de transactions concernées (pas de doublons).
3. **Anti-jointure** : parmi les clients actifs au 1ᵉʳ semestre 2024, lesquels n'ont **aucune** transaction au 2ᵉ semestre (clients perdus) ? Calculez le taux d'attrition (%) et le CA du 1ᵉʳ semestre qu'ils représentaient.
4. **Many-to-many** : montrez à l'aide d'un exemple chiffré ce qui se passe quand on joint `tx_df` (limité à 1 000 lignes) à lui-même sur `produit` (explosion), puis corrigez l'approche pour obtenir la paire « produit – produit le plus acheté dans le même magasin » sans explosion.

### Exercice 9 — Optimiser une analyse lente (profilage)
*Niveau : 🔴 avancé*

Voici une fonction naïve qui calcule, pour chaque magasin, le CA des **transactions de plus de 5 000 FCFA**, et le CA du **meilleur client** :

```python
def analyse_naive(df):
    res = []
    for m in df["magasin"].unique():
        sous = df[df["magasin"] == m]
        ca_gros = 0
        for _, ligne in sous.iterrows():
            if ligne["montant"] > 5000:
                ca_gros += ligne["montant"]
        par_client = {}
        for _, ligne in sous.iterrows():
            par_client[ligne["client"]] = par_client.get(ligne["client"], 0) + ligne["montant"]
        res.append((m, ca_gros, max(par_client.values())))
    return pd.DataFrame(res, columns=["magasin", "ca_gros", "ca_meilleur_client"])
```
1. Mesurez-la sur un échantillon de 20 000 lignes de `tx_df` (`tx_df.sample(20000, random_state=0)`) et extrapolez le temps pour 800 000 lignes.
2. Écrivez `analyse_rapide(df)` donnant **exactement** le même résultat, entièrement vectorisée (aucune boucle `for`, pas de `iterrows`/`apply`).
3. Vérifiez l'égalité (`pd.testing.assert_frame_equal`, en gérant l'ordre et les types) et calculez le facteur d'accélération sur l'échantillon.
4. Appliquez `analyse_rapide` à `tx_df` complet et donnez le temps.
5. Listez trois erreurs de performance présentes dans la version naïve.

---
# PARTIE III — Nettoyage complexe et transformations avancées

In [ ]:
# Expéditions logistiques : données JSON imbriquées, incohérentes, avec unités mélangées
rng_x = np.random.default_rng(500)
pays_liste = ["Bénin", "Togo", "Ghana", "Nigeria", "Côte d'Ivoire"]
statuts = ["LIVRÉ", "livré", "En transit", "en transit", "Retardé", "RETARDÉ ", "Perdu"]
expeditions = []
for i in range(4000):
    nb_colis = int(rng_x.integers(0, 5))
    depart = pd.Timestamp("2024-01-01") + pd.Timedelta(days=int(rng_x.integers(0, 330)))
    duree = int(rng_x.integers(-2, 12))
    rec = {
        "id": f"EXP-{i:05d}",
        "client": {"nom": f"Client {int(rng_x.integers(1, 400))}", "pays": str(rng_x.choice(pays_liste)),
                   "contact": {"tel": f"+229 {int(rng_x.integers(60000000, 99999999))}"}},
        "colis": [],
        "dates": {"depart": depart.strftime("%Y-%m-%d"),
                  "arrivee": (depart + pd.Timedelta(days=duree)).strftime("%d/%m/%Y") if rng_x.random() > 0.15 else None},
        "statut": str(rng_x.choice(statuts)),
    }
    for _ in range(nb_colis):
        kg = float(rng_x.lognormal(1.0, 0.8))
        poids = f"{kg:.1f} kg" if rng_x.random() < 0.7 else f"{int(kg * 1000)} g"
        val = int(round(rng_x.lognormal(10, 0.8), -2))
        valeur = f"{val:,} FCFA".replace(",", " ") if rng_x.random() < 0.8 else f"{val / 655.957:.0f} EUR"
        rec["colis"].append({"poids": poids, "valeur": valeur})
    if rng_x.random() < 0.03:
        del rec["client"]
    if "client" in rec and rng_x.random() < 0.05:
        rec["client"]["contact"] = None
    expeditions.append(rec)
print(len(expeditions)); print(expeditions[0])

### Exercice 10 — Aplatir et nettoyer un JSON imbriqué
*Niveau : 🔴 avancé*

`expeditions` est une liste de 4 000 dictionnaires imbriqués (client → contact, liste de colis, dates, statut). Certains champs sont absents, nuls ou dans des formats différents.

1. Construisez `colis` : **une ligne par colis** avec les colonnes `id_exp`, `pays`, `statut`, `poids_kg`, `valeur_fcfa`, `date_depart`, `date_arrivee`. Utilisez `pd.json_normalize` avec `record_path` et `meta` — en gérant les expéditions sans colis et celles dont la clé `client` est absente (indice : `errors="ignore"`).
2. `poids_kg` : convertir « 2.5 kg » et « 800 g » en kilogrammes (float).
3. `valeur_fcfa` : convertir « 12 000 FCFA » en nombre, et « 18 EUR » en FCFA au taux fixe 655,957.
4. `statut` : normaliser (casse, espaces) en 4 modalités : `Livré`, `En transit`, `Retardé`, `Perdu`.
5. Dates : `date_depart` au format ISO, `date_arrivee` au format `jj/mm/aaaa` (ou absente). Calculez `duree_j`. Identifiez les **incohérences** (arrivée avant le départ) et les expéditions livrées sans date d'arrivée.
6. Produisez un **rapport de complétude** : pourcentage de valeurs manquantes par colonne de `colis`.
7. Par pays : nombre de colis, poids total (kg), valeur médiane (FCFA), durée médiane de livraison (jours) en excluant les incohérences.

In [ ]:
# Ventes mensuelles en format « large » : 60 produits x 24 mois (2023-01 .. 2024-12), avec saisonnalités différentes
rng_w = np.random.default_rng(600)
mois_cols = pd.period_range("2023-01", "2024-12", freq="M").astype(str)
n_p = 60
niveau = rng_w.lognormal(np.log(400), 0.6, n_p)
phase = rng_w.integers(0, 12, n_p)
amp = rng_w.uniform(0.1, 0.5, n_p)
t = np.arange(24)
tendance = 1 + rng_w.normal(0.01, 0.01, n_p)[:, None] * t
saison = 1 + amp[:, None] * np.sin(2 * np.pi * (t[None, :] - phase[:, None]) / 12)
ventes_large = pd.DataFrame(
    np.round(niveau[:, None] * tendance * saison * rng_w.lognormal(0, 0.08, (n_p, 24))).astype(int),
    columns=mois_cols)
ventes_large.insert(0, "famille", rng_w.choice(["Textile", "Cosmétique", "Épicerie", "Hygiène"], n_p))
ventes_large.insert(0, "produit", [f"PRD-{i:03d}" for i in range(n_p)])
ventes_large.head()

### Exercice 11 — Du format large au format long, et retour
*Niveau : 🔴 avancé*

`ventes_large` contient une ligne par produit et une colonne par mois (`"2023-01"` … `"2024-12"`).

1. Passez au format **long** (`produit`, `famille`, `mois` en `Period`, `ventes`) avec `melt`.
2. Calculez pour chaque (produit, mois) de 2024 la **croissance annuelle** (YoY, %) par rapport au même mois 2023, avec un `groupby(...).shift(12)` sur le format long trié, **et** via un calcul direct sur le format large (comparez les deux).
3. Calculez la **moyenne mobile 3 mois** par produit et le **rang** de chaque produit parmi les 60 pour chaque mois.
4. Calculez un **indice de saisonnalité** pour chaque produit : (moyenne des ventes du mois calendaire) / (moyenne annuelle du produit), sur les 24 mois. Pour chaque produit, quel est le mois de pic ?
5. Reconstruisez un tableau large à **colonnes hiérarchiques** `(année, mois)` avec `pivot_table`, puis calculez par famille le total annuel et sa variation 2024/2023.
6. Quelles familles et quels produits sont en croissance de plus de 15 % ? Combien de produits sont « tirés » par un seul mois (mois de pic > 40 % de leur total annuel) ?

In [ ]:
# Panel de 40 stations de mesure de la qualité de l'air (PM2.5, µg/m³) sur 2024 : saisonnalité, tendance propre à chaque station, bruit autocorrélé, trous
rng_s = np.random.default_rng(700)
jours_s = pd.date_range("2024-01-01", "2024-12-31", freq="D")
n_st = 40
stations = [f"ST-{i:02d}" for i in range(n_st)]
niv = rng_s.uniform(20, 70, n_st)
pente = rng_s.normal(0, 0.03, n_st); pente[[3, 17, 28]] += 0.12                # 3 stations en dégradation nette
sais = 15 * np.cos(2 * np.pi * (np.arange(len(jours_s)) - 20) / 366)           # pic en saison sèche (janvier)
bruit = np.zeros((len(jours_s), n_st))
eps = rng_s.normal(0, 6, (len(jours_s), n_st))
for k in range(1, len(jours_s)):
    bruit[k] = 0.6 * bruit[k - 1] + eps[k]
pm25 = niv[None, :] + pente[None, :] * np.arange(len(jours_s))[:, None] + sais[:, None] + bruit
pm25 = np.clip(pm25, 2, None).round(1)
pm25[rng_s.random(pm25.shape) < 0.03] = np.nan
pm25[100:112, 5] = np.nan
air = pd.DataFrame(pm25, index=jours_s, columns=stations).rename_axis("date").reset_index().melt(
    id_vars="date", var_name="station", value_name="pm25")
print(air.shape, air["pm25"].isna().mean().round(3))

### Exercice 12 — Panel de séries temporelles : tendances, alertes et autocorrélation
*Niveau : 🔴 avancé*

`air` (format long : date × station) contient 40 × 366 mesures journalières de PM2.5 avec des valeurs manquantes.

1. Passez en format **large** `(366 dates × 40 stations)`. Quels sont les taux de valeurs manquantes par station ? Quelles stations ont une période manquante de plus de 7 jours consécutifs ?
2. Comblez les trous de **5 jours ou moins** par interpolation temporelle (par station), sans toucher aux trous plus longs.
3. Calculez, **pour toutes les stations en une seule opération matricielle**, la pente de la tendance linéaire (µg/m³ par jour) et sa significativité (statistique t) — moindres carrés sur `t = 0..365`, en ignorant les NaN restants (indice : travaillez sur les masques ou comblez ; justifiez votre choix). Quelles stations ont une tendance significativement positive (t > 3) ? Attendu : stations 3, 17 et 28 en tête.
4. Calculez le **z-score glissant** par station (fenêtre de 30 jours) et comptez, par mois, le nombre de jours d'alerte (z > 2) toutes stations confondues.
5. Pour la station `ST-00`, calculez l'**autocorrélation** aux décalages 1 à 10 jours (sans `statsmodels`), après avoir retiré la saisonnalité annuelle et la tendance. Que remarquez-vous sur les premières valeurs ?
6. Construisez les variables retardées `lag1`, `lag7` et la moyenne des 7 jours précédents (sans fuite : la valeur du jour exclue), par station, sur le format long.

---
# PARTIE IV — Statistique appliquée

### Exercice 13 — Le paradoxe de Simpson (calculs de calcul rénal)
*Niveau : 🔴 avancé*

Données classiques (Charig et al., 1986) : deux traitements A et B de calculs rénaux, selon la taille du calcul.

| | Petits calculs | Gros calculs |
|---|---|---|
| Traitement A | 81 succès sur 87 | 192 succès sur 263 |
| Traitement B | 234 succès sur 270 | 55 succès sur 80 |

1. Construisez un DataFrame `patients` de **une ligne par patient** (`traitement`, `taille`, `succes` ∈ {0,1}) à partir de ces effectifs (utilisez `np.repeat`).
2. Calculez le taux de succès de chaque traitement **globalement**, puis **par taille de calcul**. Quel traitement semble meilleur globalement ? Dans chaque strate ?
3. Expliquez le paradoxe en calculant la **répartition des tailles de calcul** selon le traitement. Quelle variable cachée (de confusion) intervient ?
4. Calculez l'**odds ratio de Mantel-Haenszel** (estimation combinée sur les deux strates) de A contre B et un intervalle de confiance à 95 % par **bootstrap** (2 000 rééchantillonnages au niveau des patients, calcul vectorisé de la statistique).
5. Que conclure ? Quelle analyse cause-effet faudrait-il pour trancher (randomisation) ?

### Exercice 14 — Test A/B : p-valeur, puissance et permutation vectorisée
*Niveau : 🔴 avancé*

Une page de paiement a deux versions : A (contrôle) a converti 528 visiteurs sur 4 800 ; B (variante) en a converti 627 sur 5 100.

1. Calculez les taux, la différence et un **test z** de comparaison de deux proportions (variance poolée) à la main ; comparez la p-valeur à `scipy.stats.chi2_contingency(correction=False)`.
2. Intervalle de confiance à 95 % de la différence de taux (méthode de Wald, variance non poolée).
3. **Test de permutation** : sous H₀, les étiquettes A/B sont interchangeables. Estimez la p-valeur par 100 000 permutations en exploitant le fait que le nombre de convertis tombant dans le groupe A suit une **loi hypergéométrique** (`rng.hypergeometric`), puis vérifiez avec un vrai test de permutation de 10 000 mélanges de l'échantillon complet.
4. Quel est l'effet minimal détectable (MDE) avec 80 % de puissance au seuil de 5 % pour n = 4 800 par groupe, en partant d'un taux de base de 11 % ? Vérifiez **par simulation** (2 000 expériences simulées avec `rng.binomial`) que la puissance vaut bien ≈ 80 % pour ce MDE, puis calculez la puissance pour l'effet observé (+1,3 point). Que conclure sur la fiabilité d'un résultat « tout juste significatif » ?
5. Si l'on regarde les résultats chaque jour pendant 20 jours en s'arrêtant dès que p < 0,05, quel est le taux réel de faux positifs en l'absence d'effet ? (simulez 5 000 expériences vides, 20 « coups d'œil ») Quel enseignement en tirer ?

---
# PARTIE V — Pipelines de préparation et algèbre linéaire appliquée

In [ ]:
# 50 000 inscriptions imparfaites pour le pipeline de préparation
rng_q = np.random.default_rng(800)
M = 50_000
pays_q = np.array(["Bénin", "benin", "BENIN", "Togo", "togo", "Côte d'Ivoire", "Cote d'Ivoire", "Ghana", "ghana", "Nigeria"])
dom = np.array(["gmail.com", "yahoo.fr", "outlook.com", "gmail.com", "hotmail.com"])
users = np.array([f"user{i}" for i in range(5000)])
emails = np.char.add(np.char.add(users[rng_q.integers(0, 5000, M)], "@"), dom[rng_q.integers(0, 5, M)])
emails = np.where(rng_q.random(M) < 0.04, "pas-un-email", emails)
d0 = pd.Timestamp("2024-01-01") + pd.to_timedelta(rng_q.integers(0, 300, M), unit="D")
inscriptions = pd.DataFrame({
    "id": np.arange(M),
    "email": emails,
    "pays": pays_q[rng_q.integers(0, len(pays_q), M)],
    "date": np.where(rng_q.random(M) < 0.7, d0.strftime("%Y-%m-%d"), d0.strftime("%d/%m/%Y")),
    "age": np.where(rng_q.random(M) < 0.02, rng_q.choice([-3, 0, 150, 999], M), np.clip(rng_q.normal(32, 10, M), 16, 80).round()),
    "montant": np.where(rng_q.random(M) < 0.05, None, np.round(rng_q.lognormal(8.5, 0.7, M), 0)).astype(object),
})
inscriptions = pd.concat([inscriptions, inscriptions.sample(500, random_state=2)], ignore_index=True)
inscriptions.loc[rng_q.choice(len(inscriptions), 300, replace=False), "montant"] = "n/a"
print(inscriptions.shape)

### Exercice 15 — Construire un pipeline de nettoyage testable et traçable
*Niveau : 🔴 avancé*

Concevez un **pipeline de préparation** pour `inscriptions` composé de fonctions pures (qui reçoivent un DataFrame et en renvoient un nouveau, sans modifier l'entrée), chaînées avec `.pipe`.

Étapes minimales : `dedoublonner` (sur `id`), `normaliser_pays` (casse, accents → 5 pays canoniques), `parser_dates` (deux formats), `valider_emails` (regex, ajoute `email_valide`), `nettoyer_age` (hors [16, 100] → NaN), `convertir_montant` (numérique, « n/a » → NaN), `ajouter_features` (mois d'inscription, jour de la semaine, `domaine` de l'e-mail).

Exigences :
1. Un **décorateur** `tracer` qui consigne, pour chaque étape, le nombre de lignes avant/après et la durée, dans une liste globale `journal`.
2. Une fonction `valider(df)` qui lève des `AssertionError` explicites (unicité des `id`, pays dans la liste canonique, dates non nulles, `age` entre 16 et 100 ou NaN, etc.).
3. Montrez que le pipeline est **idempotent** : `pipeline(pipeline(df))` donne le même résultat que `pipeline(df)` (utilisez `DataFrame.equals`).
4. Montrez qu'il **ne modifie pas** l'entrée (`inscriptions` identique avant/après).
5. Écrivez deux **tests unitaires** (de simples `assert`) sur des mini-DataFrames fabriqués à la main pour `normaliser_pays` et `valider_emails`.
6. Affichez le journal sous forme de DataFrame et un rapport final de qualité (taux de valeurs manquantes par colonne, nombre d'e-mails invalides, pays).

### Exercice 16 — ACP à la main : valeurs propres, SVD et reconstruction
*Niveau : 🔴 avancé*

Les données `X` (500 observations × 8 variables) proviennent de 3 facteurs latents + bruit :

```python
rng_pca = np.random.default_rng(900)
facteurs = rng_pca.normal(size=(500, 3)) * np.array([3.0, 2.0, 1.2])
charges = rng_pca.normal(size=(3, 8))
X = facteurs @ charges + rng_pca.normal(0, 0.5, (500, 8)) + np.arange(8) * 5     # variables d'échelles/offsets divers
```
1. Centrez-réduisez `X` en `Z`. Calculez la matrice de covariance (= corrélation) de `Z`, ses valeurs/vecteurs propres avec `np.linalg.eigh`, triés par valeur propre décroissante. Donnez la part de variance expliquée et la variance cumulée.
2. Combien de composantes pour 90 % de la variance ? Cela correspond-il au nombre de facteurs latents (3) ?
3. Projetez les données sur les `k` premières composantes puis **reconstruisez-les** ; calculez l'erreur quadratique moyenne de reconstruction pour `k = 1…8` **sans boucle sur k** (indice : l'erreur pour `k` composantes est la somme des variances des composantes écartées, à vérifier numériquement).
4. Obtenez les mêmes composantes par la **décomposition en valeurs singulières** de `Z` (`np.linalg.svd`) et vérifiez la correspondance (les vecteurs sont définis au signe près) et la relation `λ = s² / (n − 1)`.
5. Si `scikit-learn` est installé, comparez avec `sklearn.decomposition.PCA` (sinon, passez cette question).
6. Les « charges » de la 1ʳᵉ composante : quelles variables contribuent le plus ? Que se passe-t-il si vous oubliez de réduire les données (centrage seul) ?

In [ ]:
# Deux séries de prix corrélées sur 5 000 pas de temps, avec une relation qui change au cours du temps
rng_f = np.random.default_rng(1000)
T = 5000
base = np.cumsum(rng_f.normal(0, 1, T))
rho = np.where(np.arange(T) < 2500, 0.2, 0.8)                           # la corrélation passe de 0,2 à 0,8 à mi-parcours
r1 = rng_f.normal(size=T)
r2 = rho * r1 + np.sqrt(1 - rho ** 2) * rng_f.normal(size=T)
prix1 = 100 + np.cumsum(r1) * 0.5
prix2 = 80 + np.cumsum(r2) * 0.4
# motif à retrouver : un « creux puis rebond » inséré à des endroits connus
motif = np.concatenate([np.linspace(0, -3, 20), np.linspace(-3, 1, 20)])
prix_motif = 50 + np.cumsum(rng_f.normal(0, 0.3, T))
for pos in (700, 2100, 4200):
    prix_motif[pos:pos + 40] = prix_motif[pos] + motif + rng_f.normal(0, 0.08, 40)
print(prix1[:3].round(2), prix_motif.shape)

### Exercice 17 — Fenêtres glissantes vectorisées : corrélation, drawdown et recherche de motif
*Niveau : 🔴 avancé*

Avec `sliding_window_view` (`from numpy.lib.stride_tricks import sliding_window_view`) :

1. Calculez la **corrélation glissante** sur 250 pas entre les *rendements* de `prix1` et `prix2` **sans boucle ni `rolling`** (via fenêtres et formules vectorisées). Vérifiez sur 3 fenêtres avec `np.corrcoef`. Repérez à quel moment la corrélation passe de ≈ 0,2 à ≈ 0,8 (détection du changement de régime).
2. Calculez le **drawdown maximal** glissant de `prix1` sur 500 pas (plus forte baisse entre un plus haut et un plus bas ultérieur dans la fenêtre), sans boucle Python sur les fenêtres (indice : `np.maximum.accumulate` le long de l'axe des fenêtres).
3. **Recherche de motif** : dans `prix_motif`, retrouvez les trois occurrences du motif `motif` (40 points) en comparant chaque fenêtre de 40 points après **normalisation z-score de chaque fenêtre** (invariante au niveau et à l'échelle). Donnez les 3 meilleures positions (non chevauchantes) et vérifiez qu'elles sont proches de 700, 2100 et 4200.
4. Mémoire : combien d'octets occupe la matrice de fenêtres `(4961, 40)` ? Est-ce une copie ou une vue ? Que devient la mémoire si vous la modifiez ?

---
# PARTIE VI — MINI-PROJETS D'ANALYSE DE DONNÉES

Deux projets complets, dans l'esprit d'une vraie mission : un jeu de données imparfait, des questions ouvertes, des choix méthodologiques à justifier et des recommandations à formuler.

**Livrables attendus pour chaque projet** : un notebook propre (vous pouvez le faire dans de nouvelles cellules sous le projet) avec : (1) un **contrôle qualité** documenté ; (2) les analyses demandées avec leurs vérifications ; (3) au moins **trois visualisations ou tableaux de synthèse** lisibles ; (4) **trois recommandations chiffrées** et leurs limites.

Évaluez votre travail avec cette grille : *les données sont-elles propres et les choix de nettoyage justifiés ? les calculs sont-ils vectorisés et vérifiés ? les conclusions sont-elles chiffrées, nuancées et honnêtes sur leurs limites ?*

---
## Projet 1 — Mobilité urbaine : les moto-taxis de Cotonou

Une application de réservation de moto-taxis (« zémidjans ») vous confie une année de courses (2024) et la météo quotidienne. La direction veut comprendre **la demande, les revenus des conducteurs et les tensions entre offre et demande**, et préparer une politique de tarification.

Tables :
- `courses` : une ligne par demande de course (terminée ou annulée) ;
- `meteo` : pluie journalière (mm) ;
- `zones` : les 12 zones de la ville avec leurs coordonnées approximatives (km).

In [ ]:
rng_m = np.random.default_rng(2024)
zones = pd.DataFrame({
    "zone": ["Cadjèhoun", "Akpakpa", "Fidjrossè", "Haie Vive", "Dantokpa", "Vodjè", "Ganhi", "Jonquet", "Gbégamey", "Agla", "Togbin", "Sènadé"],
    "x": [0.0, 6.5, -5.0, 1.5, 3.8, 5.2, 3.0, 2.2, 1.0, -2.0, -9.0, 8.0],
    "y": [0.0, 1.0, 1.2, 0.5, -0.5, 2.0, -1.5, -0.8, 2.2, 2.5, 1.0, 3.5],
    "poids_demande": [0.10, 0.12, 0.08, 0.07, 0.15, 0.06, 0.08, 0.06, 0.07, 0.09, 0.05, 0.07],
})
dates_m = pd.date_range("2024-01-01", "2024-12-31", freq="D")
saison_pluie = np.exp(-((dates_m.dayofyear.to_numpy() - 180) ** 2) / (2 * 40 ** 2))             # grande saison des pluies ~ juin
pluie = np.where(rng_m.random(len(dates_m)) < 0.08 + 0.45 * saison_pluie, rng_m.gamma(1.8, 9.0, len(dates_m)), 0.0).round(1)
meteo = pd.DataFrame({"date": dates_m, "pluie_mm": pluie})

# demande journalière : semaine / week-end, pluie => +20 % de demandes
facteur_jour = np.where(dates_m.dayofweek >= 5, 0.78, 1.0)
lam = 690 * facteur_jour * (1 + 0.2 * (pluie > 5))
n_jour = rng_m.poisson(lam)
jours_rep = np.repeat(dates_m.to_numpy(), n_jour)
n_c = len(jours_rep)
profil_h = np.array([0.4, 0.2, 0.15, 0.15, 0.3, 1.0, 3.0, 5.2, 4.8, 3.2, 2.8, 3.0, 3.6, 3.2, 2.8, 3.0, 3.8, 5.4, 5.8, 4.2, 3.0, 2.0, 1.2, 0.7])
heure = rng_m.choice(24, n_c, p=profil_h / profil_h.sum())
minute = rng_m.integers(0, 3600, n_c)
ts_m = pd.to_datetime(jours_rep) + pd.to_timedelta(heure * 3600 + minute, unit="s")

dep = rng_m.choice(len(zones), n_c, p=zones["poids_demande"] / zones["poids_demande"].sum())
arr = rng_m.choice(len(zones), n_c, p=zones["poids_demande"] / zones["poids_demande"].sum())
xy = zones[["x", "y"]].to_numpy()
dist_vol = np.linalg.norm(xy[dep] - xy[arr], axis=1)
distance = np.round(np.maximum(dist_vol, 0.8) * rng_m.uniform(1.15, 1.45, n_c), 1)
vitesse = np.where(np.isin(heure, [7, 8, 9, 17, 18, 19]), 17.0, 26.0) * rng_m.uniform(0.8, 1.2, n_c)
duree = np.round(distance / vitesse * 60 + rng_m.gamma(2.0, 1.2, n_c) + 2, 1)
pluie_c = pd.Series(pluie, index=dates_m).reindex(pd.to_datetime(jours_rep)).to_numpy()
prix = np.round((300 + 140 * distance) * (1 + 0.15 * (pluie_c > 5)) / 50) * 50

p_annul = 0.06 + 0.05 * (pluie_c > 5) + 0.05 * np.isin(heure, [7, 8, 18, 19]) + 0.03 * (zones["zone"].to_numpy()[dep] == "Dantokpa")
u = rng_m.random(n_c)
statut = np.where(u < p_annul * 0.6, "Annulée client", np.where(u < p_annul, "Annulée conducteur", "Terminée"))

ncond = 600
niveau_cond = rng_m.lognormal(0, 0.45, ncond)                      # activité inégale des conducteurs
conducteur = rng_m.choice(np.arange(1, ncond + 1), n_c, p=niveau_cond / niveau_cond.sum())
note = np.where(rng_m.random(n_c) < 0.35, np.clip(np.round(rng_m.normal(4.4, 0.8, n_c)), 1, 5), np.nan)

courses = pd.DataFrame({"id_course": np.arange(1, n_c + 1), "ts": ts_m, "id_conducteur": conducteur,
                        "zone_depart": zones["zone"].to_numpy()[dep], "zone_arrivee": zones["zone"].to_numpy()[arr],
                        "distance_km": distance, "duree_min": duree, "prix_fcfa": prix, "statut": statut,
                        "paiement": rng_m.choice(["Mobile Money", "Espèces"], n_c, p=[0.62, 0.38]), "note": note})
courses.loc[courses["statut"] != "Terminée", ["distance_km", "duree_min", "note"]] = np.nan
courses.loc[courses["statut"] != "Terminée", "prix_fcfa"] = 0
# imperfections
courses = pd.concat([courses, courses.sample(1200, random_state=1)], ignore_index=True)          # doublons
courses = courses.sort_values("ts").reset_index(drop=True)
i1 = rng_m.choice(len(courses), 300, replace=False); courses.loc[i1, "duree_min"] = -courses.loc[i1, "duree_min"].abs()
i2 = rng_m.choice(len(courses), 120, replace=False); courses.loc[i2, "distance_km"] = courses.loc[i2, "distance_km"] * 10
print(courses.shape, meteo.shape, zones.shape)
courses.head()

### Exercice 18 — PROJET 1 — Moto-taxis de Cotonou : analyse complète
*Niveau : 🏁 mini-projet*

Réalisez l'analyse en suivant ces **sept étapes**. Chaque étape doit produire un résultat chiffré vérifié.

**Étape 1 — Contrôle qualité et nettoyage.** Détectez et traitez : les doublons (de quelle nature ? sur quelle clé ?), les durées négatives, les distances aberrantes (GPS), le traitement des courses annulées (leurs prix et distances sont-ils cohérents ?). Documentez ce que vous supprimez/corrigez et pourquoi. Produisez `c` (courses nettoyées) et `t` (courses terminées).

**Étape 2 — Structure de la demande.** Construisez le tableau `heure × jour de la semaine` du nombre moyen de demandes (par jour calendaire) ; identifiez les heures de pointe. Quantifiez l'effet de la pluie (> 5 mm) sur le nombre de demandes **à jour de semaine égal** (comparez les moyennes pluie/sec pour chaque jour de la semaine puis combinez).

**Étape 3 — Tensions offre/demande.** Pour chaque zone de départ et pour chaque tranche (pointe : 7-9 h et 17-19 h / hors pointe), calculez le taux d'annulation (client + conducteur) et le temps moyen d'une course. Quelles zones et tranches sont en tension ? L'écart d'annulation entre la zone la pire et la moyenne est-il statistiquement significatif ? (test sur proportions)

**Étape 4 — Revenus des conducteurs.** Construisez une table par conducteur : nombre de courses terminées, revenu total, revenu par course, heures de conduite (somme des durées), revenu par heure. Mesurez l'inégalité (coefficient de Gini, part du revenu des 10 % les plus actifs) en NumPy. Repérez les conducteurs dont le **taux d'annulation conducteur** est anormalement élevé (z-score binomial > 3, avec au moins 100 courses attribuées).

**Étape 5 — Tarification.** Estimez par moindres carrés (NumPy uniquement) le prix en fonction de la distance, d'un indicateur de pluie et de l'heure de pointe : `prix ~ 1 + distance + pluie + pointe`. Interprétez les coefficients (tarif de base, FCFA/km, majoration pluie). Sont-ils cohérents avec la règle de tarification réelle (300 + 140/km, +15 % sous la pluie) ? Analysez les résidus.

**Étape 6 — Prévision.** Prévoyez le nombre de demandes **par heure** pour les 4 dernières semaines de l'année. Comparez deux méthodes simples : (a) « même heure, même jour, semaine précédente » ; (b) profil moyen (jour de la semaine × heure) appris sur les semaines d'avant. Évaluez par MAE et MAPE (attention aux heures creuses).

**Étape 7 — Recommandations.** Formulez trois recommandations chiffrées (par exemple : renfort d'offre dans telle zone à telle heure, majoration pluie, programme pour les conducteurs à fort taux d'annulation) avec une estimation d'impact et les limites de l'analyse.

---
## Projet 2 — Agriculture : qu'est-ce qui fait varier les rendements ?

Une coopérative agricole vous fournit le suivi de **600 exploitations sur 6 campagnes** (2019-2024) et les précipitations de la saison par région. Elle veut savoir **quels facteurs expliquent les rendements** (pluie, engrais, semences, irrigation), **quelles exploitations sont les plus performantes à conditions égales**, et **quel gain attendre d'un programme de semences améliorées**.

Tables : `fermes` (caractéristiques fixes), `campagnes` (une ligne par ferme et par campagne), `pluviometrie` (pluie de saison par région et année).

In [ ]:
rng_a = np.random.default_rng(3030)
n_f = 600
regions = ["Nord", "Centre", "Sud", "Est", "Ouest"]
fermes = pd.DataFrame({
    "id_ferme": np.arange(1, n_f + 1),
    "region": rng_a.choice(regions, n_f, p=[0.25, 0.2, 0.2, 0.15, 0.2]),
    "taille_ha": np.round(rng_a.lognormal(np.log(3.0), 0.7, n_f), 1).clip(0.3, 40),
    "sol": rng_a.choice(["sableux", "limoneux", "argileux"], n_f, p=[0.3, 0.45, 0.25]),
})
fermes["irrigation"] = rng_a.random(n_f) < (0.1 + 0.25 * (fermes["taille_ha"] > 5))
competence = rng_a.normal(0, 0.18, n_f)                                           # facteur propre à chaque ferme (inobservé)
fermes["competence_cachee"] = competence                                           # (colonne « secrète » : pour le contrôle du résultat)

annees = np.arange(2019, 2025)
pluvio = pd.DataFrame([(r, a) for r in regions for a in annees], columns=["region", "annee"])
base_pluie = {"Nord": 750, "Centre": 950, "Sud": 1200, "Est": 900, "Ouest": 1000}
pluvio["pluie_mm"] = [round(base_pluie[r] + rng_a.normal(0, 130)) for r in pluvio["region"]]

camp = fermes.assign(key=1).merge(pd.DataFrame({"annee": annees, "key": 1}), on="key").drop(columns="key")
camp = camp.merge(pluvio, on=["region", "annee"])
adopt = 1 / (1 + np.exp(-(-2.0 + 0.30 * camp["taille_ha"] + 0.35 * (camp["annee"] - 2019))))      # les grandes fermes adoptent plus et plus tôt
camp["semences"] = np.where(rng_a.random(len(camp)) < adopt, "améliorées", "traditionnelles")
camp["culture"] = rng_a.choice(["Maïs", "Soja", "Coton"], len(camp), p=[0.5, 0.25, 0.25])
camp["engrais_kg_ha"] = np.clip(rng_a.gamma(3.0, 18.0, len(camp)) * (1 + 0.3 * (camp["taille_ha"] > 5)), 0, 250).round()

base_c = camp["culture"].map({"Maïs": 2.2, "Soja": 1.4, "Coton": 1.1})
pluie_eff = -((camp["pluie_mm"] - 1000) / 600) ** 2 * (1 - 0.6 * camp["irrigation"])           # optimum vers 1 000 mm, irrigation atténue
engrais_eff = 0.9 * (1 - np.exp(-camp["engrais_kg_ha"] / 80))                                   # rendements décroissants
sol_eff = camp["sol"].map({"sableux": -0.15, "limoneux": 0.10, "argileux": 0.0})
log_rend = (np.log(base_c) + pluie_eff + 0.35 * engrais_eff + sol_eff + 0.10 * np.log(camp["taille_ha"])
            + np.log(1.15) * (camp["semences"] == "améliorées") + camp["competence_cachee"] + rng_a.normal(0, 0.12, len(camp)))
camp["rendement_t_ha"] = np.exp(log_rend).round(2)
camp = camp.drop(columns=["competence_cachee"])
secret = fermes.set_index("id_ferme")["competence_cachee"]
fermes = fermes.drop(columns="competence_cachee")

# imperfections
camp.loc[rng_a.choice(len(camp), 90, replace=False), "engrais_kg_ha"] = np.nan
camp.loc[rng_a.choice(len(camp), 40, replace=False), "rendement_t_ha"] *= 10                    # erreur d'unité (kg/ha vs t/ha)
camp.loc[rng_a.choice(len(camp), 25, replace=False), "rendement_t_ha"] = np.nan
camp = camp.sample(frac=1, random_state=0).reset_index(drop=True)
campagnes = camp[["id_ferme", "annee", "culture", "semences", "engrais_kg_ha", "rendement_t_ha"]].copy()
pluviometrie = pluvio.copy()
print(fermes.shape, campagnes.shape, pluviometrie.shape)
campagnes.head()

### Exercice 19 — PROJET 2 — Rendements agricoles : analyse complète
*Niveau : 🏁 mini-projet*

Suivez ces **sept étapes** :

**Étape 1 — Intégration et qualité.** Joignez `campagnes`, `fermes` et `pluviometrie` en une table `d` (une ligne par ferme-campagne ; vérifiez qu'il y a 3 600 lignes, une par couple unique). Détectez les valeurs manquantes (engrais) et les **erreurs d'unité** de rendement (valeurs ×10 : repérez-les par rapport à la distribution par culture, par ex. règle des 3×IQR ou comparaison à la médiane). Corrigez/justifiez. Imputez l'engrais par la médiane de la (culture, taille>5 ha) *calculée sans les valeurs fautives*.

**Étape 2 — Exploration.** Rendement moyen par culture, par type de semences, par irrigation et par année. Tracez (tableau) le rendement moyen par tranche de pluie (`pd.cut`, 6 classes) pour la culture Maïs : observez-vous une relation en cloche ? Mesurez la corrélation entre taille de ferme et adoption de semences améliorées.

**Étape 3 — Modèle explicatif par moindres carrés (NumPy).** Estimez `log(rendement)` en fonction de : constante, culture (2 indicatrices), semences améliorées, irrigation, `engrais`, `engrais²`, `pluie`, `pluie²`, sol (2 indicatrices), `log(taille)`. Donnez coefficients, erreurs-types, statistiques t, R² et RMSE (en vous aidant de `np.linalg.inv((XᵀX))` pour la matrice de variance, ou d'une décomposition). Interprétez : l'effet des semences améliorées (en %), l'optimum de pluie.

**Étape 4 — Validation temporelle.** Entraînez sur 2019-2022, testez sur 2023-2024. Comparez le RMSE (sur `log(rendement)`) à celui d'une baseline (moyenne par culture). Le modèle généralise-t-il ?

**Étape 5 — Efficacité des fermes.** Calculez pour chaque ferme le **résidu moyen** du modèle (sa performance « à conditions égales »). Mesurez la persistance : corrélation entre l'efficacité calculée sur 2019-2021 et sur 2022-2024. Vérifiez enfin contre la compétence cachée `secret` (disponible pour contrôle) que le classement retrouvé est cohérent. Quelles sont les 10 % de fermes les plus performantes ? Dans quelles régions ?

**Étape 6 — Impact des semences améliorées.** (a) Différence brute de rendement moyen entre semences améliorées et traditionnelles ; (b) estimation ajustée (coefficient du modèle, en %) ; (c) intervalle de confiance à 95 % par **bootstrap par ferme** (rééchantillonner des fermes entières, 300 réplications). Pourquoi l'écart brut surestime-t-il l'effet réel ? (indice : variable de confusion)

**Étape 7 — Recommandations.** Estimez le gain de production (tonnes) si toutes les campagnes à semences traditionnelles de 2024 utilisaient des semences améliorées (prédiction contrefactuelle), ventilé par région. Formulez trois recommandations chiffrées et leurs limites.

---
## Bilan

Vous avez maintenant parcouru l'ensemble des compétences NumPy et Pandas : tableaux multidimensionnels, broadcasting, algèbre linéaire, vectorisation, nettoyage de données complexes, jointures, séries temporelles, statistiques appliquées, pipelines et projets complets.

**Pour aller plus loin** : passez à `../02_MACHINE_LEARNING/COURS_MACHINE_LEARNING.md`. Les modèles de Machine Learning réutilisent exactement ces outils : `X` (matrice NumPy), `y` (vecteur), préparation des données (Pandas), moindres carrés, ACP, distances, validation.

---

# CORRECTIONS ET EXPLICATIONS

> ⚠️ **Section de corrections.** Avant de lire une correction, essayez vraiment l'exercice, relisez le cours, testez des variantes. Comparez ensuite votre méthode à la solution : plusieurs solutions correctes existent souvent.

Chaque correction contient : une **solution exécutable**, l'**explication du raisonnement**, les **erreurs possibles** et, quand c'est pertinent, une **autre approche**.

Les solutions utilisent uniquement les jeux de données définis plus haut dans le notebook : exécutez d'abord toutes les cellules de préparation (menu *Run → Run All Cells* fonctionne, vos cellules de réponse vides n'affichent rien).

## Correction — Exercice 1 : Standardisation d'un lot d'images (broadcasting 4D)

In [ ]:
# 1. par (image, canal) : on réduit sur les axes 2 et 3 en gardant les dimensions
m1 = images.mean(axis=(2, 3), keepdims=True)
s1 = images.std(axis=(2, 3), keepdims=True)
z1 = (images - m1) / s1
print(np.allclose(z1.mean(axis=(2, 3)), 0), np.allclose(z1.std(axis=(2, 3)), 1))

# 2. par canal sur le lot : axes 0, 2, 3
m2 = images.mean(axis=(0, 2, 3), keepdims=True)
s2 = images.std(axis=(0, 2, 3), keepdims=True)
z2 = (images - m2) / s2
print(np.allclose(z2.mean(axis=(2, 3)), 0), abs(z2.mean(axis=(2, 3))).max().round(2))

# 3. min-max par image
mn = images.min(axis=(1, 2, 3), keepdims=True); mx = images.max(axis=(1, 2, 3), keepdims=True)
z3 = (images - mn) / (mx - mn)
print(z3.min(), z3.max(), z3.min(axis=(1, 2, 3)).max(), z3.max(axis=(1, 2, 3)).min())

# 4. transformation de couleur
M = np.array([[0.3, 0.6, 0.1], [0.5, 0.4, 0.1], [0.2, 0.2, 0.6]])
out = np.einsum("ij,njhw->nihw", M, images)
pixel = images[10, :, 5, 7]
print(np.allclose(out[10, :, 5, 7], M @ pixel))

# 5. aplatissement
flat = images.reshape(64, -1)
print(flat.shape, np.shares_memory(flat, images))

**Raisonnement.** Pour standardiser par groupe, on calcule les statistiques en **réduisant les axes qui définissent le groupe** (ici les pixels, axes 2 et 3) avec `keepdims=True`, ce qui laisse des formes `(64, 3, 1, 1)` directement diffusables contre `(64, 3, 24, 24)`. Avec une standardisation par canal sur tout le lot (axes 0, 2, 3), seules les moyennes **globales** par canal sont nulles : chaque image garde sa propre luminosité moyenne (ce qui est parfois voulu). `einsum("ij,njhw->nihw")` exprime « pour chaque image et pixel, multiplier le vecteur couleur par M ». `reshape` renvoie une vue quand c'est possible (tableau contigu en mémoire).

**Erreurs possibles.** Oublier `keepdims=True` : les formes `(64, 3)` et `(64, 3, 24, 24)` ne se diffusent pas (broadcasting aligné par la droite). Diviser par un écart-type nul (image uniforme) → NaN : en production, ajoutez un epsilon.

**Autre approche.** `np.tensordot(M, images, axes=([1], [1]))` donne le résultat avec les axes dans un autre ordre ; il faut ensuite `moveaxis`. `einsum` est plus lisible ici.

## Correction — Exercice 2 : Plus proches voisins sur 20 000 vecteurs (bloc par bloc)

In [ ]:
# 1-2
E = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
S = E @ E.T
np.fill_diagonal(S, -np.inf)                               # on exclut le document lui-même
k = 3
idx = np.argpartition(-S, k, axis=1)[:, :k]                # les k plus grands, non triés
ok = all(set(idx[i]) == set(np.argsort(-S[i])[:k]) for i in range(100))
print(S.shape, ok)

# 3
print(round((themes[idx] == themes[:, None]).mean() * 100, 1), "% de voisins du même thème")

# 4
def voisins_blocs(X, k=3, bloc=500):
    Xn = X / np.linalg.norm(X, axis=1, keepdims=True)
    out = np.empty((len(X), k), dtype=np.int32)
    for d in range(0, len(X), bloc):
        sim = Xn[d:d + bloc] @ Xn.T                        # (bloc, N)
        sim[np.arange(sim.shape[0]), np.arange(d, d + sim.shape[0])] = -np.inf
        out[d:d + bloc] = np.argpartition(-sim, k, axis=1)[:, :k]
    return out

res, t = chrono(voisins_blocs, embeddings_xl)
print(res.shape, f"{t:.2f}s")
# contrôle : plus proche voisin du premier bloc via tri complet
Xn = embeddings_xl / np.linalg.norm(embeddings_xl, axis=1, keepdims=True)
sim0 = Xn[:500] @ Xn.T
sim0[np.arange(500), np.arange(500)] = -np.inf
best1 = sim0.argmax(axis=1)
print(all(best1[i] in res[i] for i in range(500)))

**Raisonnement.** Une similarité cosinus entre vecteurs normalisés est un simple produit matriciel. `argpartition(-S, k)` place les `k` plus grandes valeurs dans les `k` premières positions en `O(n)` (au lieu du tri complet en `O(n log n)`), sans garantir leur ordre interne. Pour les 20 000 vecteurs, on traite les lignes **par blocs** : chaque bloc produit une matrice `(500, 20000)` (≈ 40 Mo en float32) qu'on réduit tout de suite à `k` indices. C'est la stratégie standard quand le résultat intermédiaire ne tient pas en mémoire.

**Erreurs possibles.** Oublier d'exclure la diagonale : chaque document est son propre voisin n°1. Utiliser `argsort` sur 20 000 × 20 000 : très lent et gourmand en mémoire. Calculer `E @ E.T` en float64 sur 20 000 lignes : 3,2 Go.

## Correction — Exercice 3 : Impôt progressif sur un million de revenus

In [ ]:
bornes = np.array([0, 800_000, 2_000_000, 4_000_000, 8_000_000, np.inf])
taux = np.array([0.0, 0.10, 0.15, 0.20, 0.30])

def impot_boucle(r):
    total = 0.0
    for lo, hi, t in zip(bornes[:-1], bornes[1:], taux):
        total += t * max(0.0, min(r, hi) - lo)
    return total

print(impot_boucle(3_000_000))                    # 270000

def impot_broadcast(r):
    r = np.asarray(r, dtype=float)[:, None]                              # (n, 1)
    part = np.clip(r - bornes[:-1], 0, bornes[1:] - bornes[:-1])         # (n, 5) : part dans chaque tranche
    return part @ taux

cumul = np.concatenate([[0.0], np.cumsum(taux[:-1] * np.diff(bornes[:-1]))])   # impôt cumulé au début de chaque tranche

def impot_searchsorted(r):
    r = np.asarray(r, dtype=float)
    i = np.searchsorted(bornes[1:-1], r, side="right")                  # indice de tranche
    return cumul[i] + (r - bornes[:-1][i]) * taux[i]

ech = revenus[:1000]
ref = np.array([impot_boucle(r) for r in ech])
print(np.allclose(ref, impot_broadcast(ech)), np.allclose(ref, impot_searchsorted(ech)))
i1, t1 = chrono(impot_broadcast, revenus)
i2, t2 = chrono(impot_searchsorted, revenus)
print(np.allclose(i1, i2), f"broadcast {t1:.3f}s (tableau 1M×5 = {1_000_000 * 5 * 8 / 1e6:.0f} Mo) | searchsorted {t2:.3f}s")

df = pd.DataFrame({"revenu": revenus, "impot": i2})
df["decile"] = pd.qcut(df["revenu"], 10, labels=range(1, 11))
d = df.groupby("decile", observed=True).agg(revenu=("revenu", "mean"), impot=("impot", "mean"))
d["taux_effectif_%"] = d["impot"] / d["revenu"] * 100
print(d.round(1))
print(round(df.loc[df["decile"] == 10, "impot"].sum() / df["impot"].sum() * 100, 1), "% de l'impôt payé par le décile supérieur")

**Raisonnement.** Deux idées vectorisées. (a) **Broadcasting** : pour chaque revenu, la part située dans chaque tranche est `clip(r − borne_basse, 0, largeur)` ; un produit matriciel avec les taux donne l'impôt. Simple, mais crée un tableau `(n, 5)`. (b) **`searchsorted`** : on précalcule l'impôt dû au début de chaque tranche (somme cumulée), puis pour chaque revenu on trouve sa tranche et on ajoute la part marginale ; mémoire `O(n)` et temps souvent inférieur. La progressivité se lit dans le taux **effectif** croissant avec le décile.

**Erreurs possibles.** Appliquer le taux de la tranche à **tout** le revenu (impôt « en cascade » erroné). Mauvais `side` dans `searchsorted` : un revenu exactement égal à une borne changerait de tranche (la valeur de l'impôt resterait continue, mais vérifiez-le).

## Correction — Exercice 4 : Simulation Monte-Carlo de rupture de stock

In [ ]:
def simuler(Q, n_scen=100_000, seed=0, jours=90, S0=300):
    Q = np.atleast_1d(Q).astype(float)                          # (nQ,)
    rng = np.random.default_rng(seed)
    dow = np.arange(jours) % 7
    lam_base = np.where(dow >= 5, 30.0, 18.0)                   # (jours,)
    aleas = np.exp(rng.normal(0, 0.25, size=(n_scen, jours)))
    demande = rng.poisson(lam_base * aleas)                     # (n_scen, jours)
    stock = np.full((n_scen, len(Q)), float(S0))                # un stock par (scénario, Q)
    perdues = np.zeros((n_scen, len(Q), jours))
    stocks = np.zeros((n_scen, len(Q), jours))
    for j in range(jours):
        if j > 0 and j % 7 == 0:
            stock = stock + Q[None, :]                           # livraison avant l'ouverture
        d = demande[:, j][:, None]                               # (n_scen, 1)
        vendu = np.minimum(stock, d)
        perdues[:, :, j] = d - vendu
        stock = stock - vendu
        stocks[:, :, j] = stock
    return perdues, stocks

perdues, stocks = simuler(150)
rupture = (perdues.sum(axis=2) > 0).squeeze()
print(round(rupture.mean(), 3), round(perdues.sum(axis=2).mean(), 1))

grille = np.arange(120, 201, 10)
perdues, stocks = simuler(grille, n_scen=20_000, seed=1)
p_rupt = (perdues.sum(axis=2) > 0).mean(axis=0)
v_perdues = perdues.sum(axis=2).mean(axis=0)
stock_moy = stocks.mean(axis=(0, 2))
livraisons = len(range(7, 90, 7))
cout = 15 * v_perdues + 0.05 * stock_moy * 90 + 40 * livraisons
res = pd.DataFrame({"Q": grille, "p_rupture": p_rupt, "ventes_perdues": v_perdues, "stock_moyen": stock_moy, "cout": cout})
print(res.round(3).to_string(index=False))
Q_opt = grille[cout.argmin()]
print("Q optimal :", Q_opt)
n = 20_000; p = p_rupt[cout.argmin()]; se = np.sqrt(p * (1 - p) / n)
print(f"IC95% de P(rupture) : [{p - 1.96 * se:.3f} ; {p + 1.96 * se:.3f}]")

**Raisonnement.** On vectorise **sur les scénarios** (et sur la grille de `Q`) et on boucle uniquement sur le temps, car le stock du jour dépend de celui de la veille (dépendance séquentielle). L'astuce de la dimension supplémentaire : le stock a la forme `(n_scen, nQ)` et la demande `(n_scen, 1)` se diffuse. `np.minimum(stock, demande)` modélise les ventes réalisées, la différence les ventes perdues. L'intervalle de confiance d'une probabilité estimée par simulation vaut `p ± 1,96 √(p(1−p)/n)` : plus on simule de scénarios, plus il se resserre.

**Erreurs possibles.** Boucler sur les 100 000 scénarios (des heures de calcul). Générer les mêmes aléas pour des `Q` différents est **voulu** ici (comparaison à aléas communs) : changer de graine à chaque `Q` ajouterait du bruit parasite entre scénarios comparés.

## Correction — Exercice 5 : Lecture par blocs : des agrégats exacts sans tout charger

In [ ]:
total = 0.0; n_tx = 0
ca_mag = pd.Series(dtype=float)
clients_vus = set()
bornes_h = np.linspace(0, tx_df["montant"].max() + 1, 2001)
hist = np.zeros(2000, dtype=np.int64)

for bloc in pd.read_csv(chemin_tx, chunksize=100_000, usecols=["magasin", "client", "montant"]):
    total += bloc["montant"].sum()
    n_tx += len(bloc)
    ca_mag = ca_mag.add(bloc.groupby("magasin")["montant"].sum(), fill_value=0)
    clients_vus.update(bloc["client"].unique())
    hist += np.histogram(bloc["montant"], bins=bornes_h)[0]

cum = np.cumsum(hist)
classe_med = np.searchsorted(cum, n_tx / 2)
mediane_approx = (bornes_h[classe_med] + bornes_h[classe_med + 1]) / 2
print(total, n_tx, len(clients_vus))
print("médiane approchée :", round(mediane_approx), "| exacte :", tx_df["montant"].median())
print(np.allclose(ca_mag.sort_index(), tx_df.groupby("magasin")["montant"].sum().sort_index()),
      total == tx_df["montant"].sum(), len(clients_vus) == tx_df["client"].nunique())

complet = pd.read_csv(chemin_tx)
leger = pd.read_csv(chemin_tx, usecols=["magasin", "produit", "client", "quantite", "montant"],
                    dtype={"magasin": "category", "produit": "category", "client": "int32", "quantite": "int8", "montant": "float32"})
m1 = complet.memory_usage(deep=True).sum() / 1e6; m2 = leger.memory_usage(deep=True).sum() / 1e6
print(f"{m1:.0f} Mo -> {m2:.0f} Mo ({(1 - m2 / m1) * 100:.0f} % d'économie)")

**Raisonnement.** Les agrégats **décomposables** (somme, compte, minimum, maximum, somme de carrés → variance) se combinent bloc par bloc. La **médiane** ne l'est pas : la médiane des médianes n'est pas la médiane globale, car elle dépend de la répartition de toutes les valeurs. On l'approxime par un histogramme à bornes fixes (précision = largeur de classe) ; des algorithmes spécialisés (t-digest, sketches) généralisent l'idée. Les clients distincts nécessitent de mémoriser les identifiants vus (un ensemble) : exact mais proportionnel au nombre de clients distincts. Les types économes (`category`, `int8`, `float32`) réduisent fortement la mémoire.

**Erreurs possibles.** Calculer des moyennes par bloc puis les moyenner (poids inégaux entre blocs). Oublier `fill_value=0` dans `add` : les magasins absents d'un bloc deviennent NaN.

## Correction — Exercice 6 : Agrégations multiples et indicateurs personnalisés

In [ ]:
g = tx_df.groupby("magasin")
bilan = g.agg(ca=("montant", "sum"), nb_tx=("id_tx", "size"), q=("quantite", "sum"),
              p90_montant=("montant", lambda s: s.quantile(0.9)), nb_clients=("client", "nunique"))
bilan["panier_moyen"] = bilan["ca"] / bilan["nb_tx"]
bilan["prix_moyen_unit"] = bilan["ca"] / bilan["q"]
bilan["jours_actifs"] = tx_df.assign(j=tx_df["ts"].dt.date).groupby("magasin")["j"].nunique()

mp = tx_df.groupby(["magasin", "produit"]).agg(q=("quantite", "sum"), ca=("montant", "sum")).reset_index()
bilan["produit_top"] = mp.sort_values(["magasin", "q"], ascending=[True, False]).drop_duplicates("magasin").set_index("magasin")["produit"]
mp["part"] = mp["ca"] / mp.groupby("magasin")["ca"].transform("sum")
bilan["hhi"] = (mp["part"] ** 2).groupby(mp["magasin"]).sum()
bilan["part_top5"] = mp.sort_values(["magasin", "part"], ascending=[True, False]).groupby("magasin")["part"].apply(lambda s: s.head(5).sum())
bilan = bilan.drop(columns="q")

# vérification à la main sur deux magasins
for m in ["MAG-00", "MAG-17"]:
    s = tx_df[tx_df["magasin"] == m]
    par_prod = s.groupby("produit")["montant"].sum()
    assert np.isclose(bilan.loc[m, "hhi"], ((par_prod / par_prod.sum()) ** 2).sum())
    assert bilan.loc[m, "produit_top"] == s.groupby("produit")["quantite"].sum().idxmax()
    assert np.isclose(bilan.loc[m, "p90_montant"], np.percentile(s["montant"], 90))
print(bilan.sort_values("hhi", ascending=False).head(5).round(3))

**Raisonnement.** Stratégie : (1) un premier `groupby.agg` pour tout ce qui est décomposable ; (2) un second groupby sur la clé fine (magasin, produit) qui sert à calculer top produit, parts et indicateurs de concentration, puis (3) on rapatrie ces résultats vers `bilan` grâce à l'**alignement sur l'index** (affectation d'une Series indexée par magasin). On évite ainsi de lancer 50 `apply` sur des sous-DataFrames. Le HHI se calcule comme la somme des carrés des parts : il vaut 1 si un seul produit fait tout le CA.

**Erreurs possibles.** Utiliser `lambda s: s.quantile(0.9)` est correct mais plus lent que `g["montant"].quantile(0.9)` (méthode native). Pour `produit_top`, un `idxmax` après `groupby` sur un MultiIndex demande de réindexer correctement ; le `sort_values + drop_duplicates` est plus sûr.

## Correction — Exercice 7 : Dernier achat par client : quelle méthode choisir ?

In [ ]:
def m_sort(d):    return d.sort_values("ts").drop_duplicates("client", keep="last")
def m_idxmax(d):  return d.loc[d.groupby("client")["ts"].idxmax()]
def m_tail(d):    return d.sort_values("ts").groupby("client").tail(1)
def m_lexsort(d):
    ordre = np.lexsort((d["ts"].to_numpy(), d["client"].to_numpy()))        # tri par client puis ts
    c = d["client"].to_numpy()[ordre]
    dernier = np.r_[c[1:] != c[:-1], True]
    return d.iloc[ordre[dernier]]

res = {}
for nom, f in [("sort+drop_dup", m_sort), ("idxmax", m_idxmax), ("tail(1)", m_tail), ("lexsort", m_lexsort)]:
    r, t = chrono(f, tx_df)
    res[nom] = (set(r["id_tx"]), t)
ref = res["sort+drop_dup"][0]
print({k: v[0] == ref for k, v in res.items()})
print(pd.Series({k: v[1] for k, v in res.items()}).sort_values().round(3))

# premier achat : idxmin / keep="first" / tail->head ; plus gros achat : idxmax sur "montant" (pas de tri nécessaire)
premier = tx_df.loc[tx_df.groupby("client")["ts"].idxmin()]
gros = tx_df.loc[tx_df.groupby("client")["montant"].idxmax()]
print(len(premier), len(gros), tx_df["client"].nunique())

s = tx_df.sort_values(["client", "ts"])
delai = s.groupby("client")["ts"].diff().dt.total_seconds() / 86400
etendue = (s.groupby("client")["ts"].max() - s.groupby("client")["ts"].min()).dt.days
print(round(delai.median(), 1), "jours de délai médian |", etendue.describe().round(1).to_dict())

**Raisonnement.** `idxmax` ne trie rien (une passe) mais renvoie des étiquettes d'index à réutiliser avec `loc` (l'index doit être unique : ici c'est le cas). `sort_values + drop_duplicates` est lisible mais trie 800 000 lignes. `lexsort` manuel est rapide, mais moins lisible. Pour le premier/le plus gros achat, `idxmin`/`idxmax` sur une autre colonne se généralisent directement, alors que la méthode « dernier élément après tri par date » demande de changer l'ordre de tri. `groupby().diff()` calcule l'écart avec la ligne précédente **du même groupe** à condition que les données soient triées par (client, date).

**Erreurs possibles.** Utiliser `groupby().last()` : il renvoie la dernière valeur **non nulle** de chaque colonne séparément, ce qui peut mélanger des lignes différentes (ce n'est pas « la dernière ligne »). Appeler `diff()` avant d'avoir trié par (client, date).

## Correction — Exercice 8 : Jointure par intervalle, jointure temporelle et anti-jointure

In [ ]:
rng_j = np.random.default_rng(400)
dates = pd.date_range("2024-01-01", "2024-12-31", freq="D")
taux = pd.DataFrame({"date": dates[dates.dayofweek < 5]})
taux["xof_par_usd"] = (600 + np.cumsum(rng_j.normal(0, 1.2, len(taux)))).round(2)
paiements = pd.DataFrame({"ts": pd.Timestamp("2024-01-01") + pd.to_timedelta(np.sort(rng_j.integers(0, 365 * 86400, 5000)), unit="s"),
                          "usd": rng_j.uniform(10, 500, 5000).round(2)})
promos = pd.DataFrame({"produit": ["P001", "P001", "P002", "P003", "P003"],
                       "debut": pd.to_datetime(["2024-02-01", "2024-02-15", "2024-03-01", "2024-01-10", "2024-06-01"]),
                       "fin":   pd.to_datetime(["2024-02-29", "2024-03-15", "2024-03-31", "2024-02-10", "2024-06-30"]),
                       "remise": [0.10, 0.20, 0.15, 0.05, 0.25]})

# 1. merge_asof (les deux côtés triés sur la clé)
p = pd.merge_asof(paiements.sort_values("ts"), taux.sort_values("date"), left_on="ts", right_on="date",
                  direction="backward", tolerance=pd.Timedelta(days=4))
p["xof"] = p["usd"] * p["xof_par_usd"]
we = p[p["ts"].dt.dayofweek >= 5].head(3)
print(p["xof_par_usd"].isna().sum(), we[["ts", "date", "xof_par_usd"]].assign(jour=we["ts"].dt.day_name()))

# 2. jointure par intervalle
cible = tx_df[tx_df["produit"].isin(["P001", "P002", "P003"])][["id_tx", "ts", "produit", "montant"]].copy()
cible["jour"] = cible["ts"].dt.normalize()
m = cible.merge(promos, on="produit", how="left")
m = m[(m["jour"] >= m["debut"]) & (m["jour"] <= m["fin"])]
meilleure = m.groupby("id_tx")["remise"].max()
cible["remise"] = cible["id_tx"].map(meilleure).fillna(0.0)
assert len(cible) == cible["id_tx"].nunique()
cible["montant_net"] = cible["montant"] * (1 - cible["remise"])
print(len(cible), (cible["remise"] > 0).sum(), cible.groupby("produit")["remise"].mean().round(3).to_dict())

# 3. anti-jointure
s1 = tx_df[tx_df["ts"] < "2024-07-01"]; s2 = tx_df[tx_df["ts"] >= "2024-07-01"]
actifs_s1 = s1["client"].unique()
j = pd.DataFrame({"client": actifs_s1}).merge(pd.DataFrame({"client": s2["client"].unique()}), on="client", how="left", indicator=True)
perdus = j.loc[j["_merge"] == "left_only", "client"]
print(len(perdus), round(len(perdus) / len(actifs_s1) * 100, 1), "% |", round(s1[s1["client"].isin(perdus)]["montant"].sum() / s1["montant"].sum() * 100, 1), "% du CA S1")

# 4. many-to-many
petit = tx_df.head(1000)
expl = petit.merge(petit, on="produit")
print(len(petit), "->", len(expl), "lignes après jointure many-to-many")
top_prod = (tx_df.groupby(["magasin", "produit"])["quantite"].sum().reset_index()
            .sort_values(["magasin", "quantite"], ascending=[True, False]).drop_duplicates("magasin")[["magasin", "produit"]]
            .rename(columns={"produit": "produit_top"}))
res = petit.merge(top_prod, on="magasin", how="left", validate="many_to_one")
print(len(res))

**Raisonnement.** `merge_asof` fait une jointure sur la **clé la plus proche** (ici « la dernière date inférieure ou égale ») ; les deux tables doivent être triées. Elle gère naturellement les week-ends (aucun taux publié → on prend celui du vendredi). Pour la jointure par intervalle, on joint d'abord sur la clé exacte puis on filtre sur la condition d'intervalle ; la gestion des recouvrements se fait ensuite en agrégeant (`max`) par identifiant de transaction. L'anti-jointure repose sur `indicator=True` et `left_only`. L'explosion many-to-many multiplie les lignes (n² par produit) : la parade est de ramener d'abord l'une des tables à une ligne par clé (`drop_duplicates` après tri, ou agrégation) et de poser `validate="many_to_one"`.

**Erreurs possibles.** `merge_asof` sur des tables non triées → erreur ou mauvais résultat. Joindre sur `ts` (horodatage) au lieu de `jour` normalisé pour l'intervalle. Oublier de dédoublonner après la jointure par intervalle : une transaction couverte par deux promotions apparaîtrait deux fois.

## Correction — Exercice 9 : Optimiser une analyse lente (profilage)

In [ ]:
def analyse_naive(df):
    res = []
    for m in df["magasin"].unique():
        sous = df[df["magasin"] == m]
        ca_gros = 0
        for _, ligne in sous.iterrows():
            if ligne["montant"] > 5000:
                ca_gros += ligne["montant"]
        par_client = {}
        for _, ligne in sous.iterrows():
            par_client[ligne["client"]] = par_client.get(ligne["client"], 0) + ligne["montant"]
        res.append((m, ca_gros, max(par_client.values())))
    return pd.DataFrame(res, columns=["magasin", "ca_gros", "ca_meilleur_client"])

def analyse_rapide(df):
    gros = df["montant"].where(df["montant"] > 5000, 0).groupby(df["magasin"]).sum()
    meilleur = df.groupby(["magasin", "client"])["montant"].sum().groupby("magasin").max()
    out = pd.DataFrame({"ca_gros": gros, "ca_meilleur_client": meilleur}).reset_index()
    return out

ech = tx_df.sample(20_000, random_state=0)
r1, t1 = chrono(analyse_naive, ech)
r2, t2 = chrono(analyse_rapide, ech)
pd.testing.assert_frame_equal(r1.sort_values("magasin").reset_index(drop=True),
                              r2.sort_values("magasin").reset_index(drop=True), check_dtype=False)
print(f"naïve {t1:.2f}s (extrapolé 800k : {t1 * 40:.0f}s) | rapide {t2:.4f}s | ×{t1 / t2:.0f}")
_, t3 = chrono(analyse_rapide, tx_df)
print(f"rapide sur 800 000 lignes : {t3:.3f}s")

**Raisonnement.** Version rapide : (1) le filtre conditionnel devient `where(cond, 0)` puis une somme groupée ; (2) le « meilleur client » est un groupby à deux niveaux (magasin, client) suivi d'un `max` par magasin. Les erreurs de la version naïve : `iterrows` (crée une Series par ligne, ~100× plus lent que le vectorisé), double parcours des mêmes données par magasin, boucle Python imbriquée, filtrage `df[df["magasin"] == m]` répété 50 fois (balayage complet à chaque fois), dictionnaire reconstruit à la main au lieu de `groupby`.

**Erreurs possibles.** Comparer deux DataFrames sans trier ni aligner les index et types : l'égalité échoue pour de mauvaises raisons. Toujours tester l'équivalence avec un outil dédié (`assert_frame_equal`).

## Correction — Exercice 10 : Aplatir et nettoyer un JSON imbriqué

In [ ]:
colis = pd.json_normalize(expeditions, record_path="colis",
                          meta=["id", "statut", ["client", "pays"], ["dates", "depart"], ["dates", "arrivee"]],
                          errors="ignore")
colis = colis.rename(columns={"id": "id_exp", "client.pays": "pays", "dates.depart": "date_depart", "dates.arrivee": "date_arrivee"})

num = colis["poids"].str.extract(r"([\d.]+)\s*(kg|g)")
colis["poids_kg"] = num[0].astype(float) * np.where(num[1] == "g", 0.001, 1.0)

v = colis["valeur"].str.extract(r"([\d\s]+)\s*(FCFA|EUR)")
montant = pd.to_numeric(v[0].str.replace(r"\s", "", regex=True))
colis["valeur_fcfa"] = (montant * np.where(v[1] == "EUR", 655.957, 1.0)).round()

colis["statut"] = (colis["statut"].str.strip().str.lower()
                   .map({"livré": "Livré", "en transit": "En transit", "retardé": "Retardé", "perdu": "Perdu"}))

colis["date_depart"] = pd.to_datetime(colis["date_depart"], format="%Y-%m-%d")
colis["date_arrivee"] = pd.to_datetime(colis["date_arrivee"], format="%d/%m/%Y")
colis["duree_j"] = (colis["date_arrivee"] - colis["date_depart"]).dt.days
colis = colis.drop(columns=["poids", "valeur"])

incoherent = colis["duree_j"] < 0
livre_sans_date = (colis["statut"] == "Livré") & colis["date_arrivee"].isna()
print("colis :", len(colis), "| incohérences :", int(incoherent.sum()), "| livrés sans date :", int(livre_sans_date.sum()))
print((colis.isna().mean() * 100).round(1).to_string())
print(colis[~incoherent].groupby("pays").agg(nb=("id_exp", "size"), poids_total=("poids_kg", "sum"),
                                              valeur_mediane=("valeur_fcfa", "median"), duree_med=("duree_j", "median")).round(1))

**Raisonnement.** `json_normalize` aplatit les structures imbriquées : `record_path` indique la liste à « exploser » (une ligne par colis) et `meta` les champs parents à répéter ; les chemins imbriqués s'écrivent en listes. `errors="ignore"` évite l'erreur quand une clé est absente. Les `str.extract` avec expressions régulières séparent nombre et unité, ce qui évite des boucles. `np.where(cond, facteur_a, facteur_b)` applique le bon facteur de conversion. Les contrôles de cohérence (arrivée < départ, statut livré sans date) font partie intégrante du nettoyage.

**Erreurs possibles.** Parcourir la liste avec une boucle Python et `append` de dictionnaires : lent et fragile. Convertir « 800 g » avec `float()` sans extraire l'unité. Oublier que `record_path` supprime les expéditions sans colis (listes vides) : pour les compter, il faut un tableau séparé des expéditions.

## Correction — Exercice 11 : Du format large au format long, et retour

In [ ]:
long = ventes_large.melt(id_vars=["produit", "famille"], var_name="mois", value_name="ventes")
long["mois"] = pd.PeriodIndex(long["mois"], freq="M")
long = long.sort_values(["produit", "mois"]).reset_index(drop=True)

# YoY : version long (shift 12) et version large
long["yoy_%"] = (long["ventes"] / long.groupby("produit")["ventes"].shift(12) - 1) * 100
a24 = ventes_large[[c for c in ventes_large.columns if c.startswith("2024")]].to_numpy()
a23 = ventes_large[[c for c in ventes_large.columns if c.startswith("2023")]].to_numpy()
yoy_large = (a24 / a23 - 1) * 100
yoy_long = long[long["mois"].dt.year == 2024].pivot(index="produit", columns="mois", values="yoy_%").to_numpy()
print("YoY concordants :", np.allclose(yoy_long, yoy_large))

long["mm3"] = long.groupby("produit")["ventes"].transform(lambda s: s.rolling(3).mean())
long["rang"] = long.groupby("mois")["ventes"].rank(ascending=False, method="min").astype(int)

long["mois_cal"] = long["mois"].dt.month
moy_mois = long.groupby(["produit", "mois_cal"])["ventes"].mean().unstack()
indice = moy_mois.div(long.groupby("produit")["ventes"].mean(), axis=0)
pic = indice.idxmax(axis=1)
print(pic.value_counts().sort_index().to_dict())

long["annee"] = long["mois"].dt.year
tab = long.pivot_table(index="famille", columns=["annee", "mois_cal"], values="ventes", aggfunc="sum")
fam = long.pivot_table(index="famille", columns="annee", values="ventes", aggfunc="sum")
fam["var_%"] = (fam[2024] / fam[2023] - 1) * 100
print(tab.shape, "\n", fam.round(1))
prod = long.pivot_table(index="produit", columns="annee", values="ventes", aggfunc="sum")
croiss = (prod[2024] / prod[2023] - 1) * 100
print((croiss > 15).sum(), "produits à plus de +15 %")
an24 = long[long["annee"] == 2024]
part_pic = an24.groupby("produit")["ventes"].max() / an24.groupby("produit")["ventes"].sum()
print((part_pic > 0.4).sum(), "produits dont un mois dépasse 40 % de l'année")

**Raisonnement.** `melt` passe du large au long ; `shift(12)` **dans chaque groupe** donne la valeur de l'année précédente à condition que le tableau soit trié par (produit, mois) et sans mois manquant. Le calcul direct sur les matrices NumPy (2024 / 2023 colonne à colonne) donne le même résultat en une opération : preuve de cohérence. L'indice de saisonnalité normalise la moyenne de chaque mois calendaire par la moyenne annuelle ; `idxmax(axis=1)` renvoie le mois de pic. Les colonnes hiérarchiques `(année, mois)` s'obtiennent en passant une liste à `columns=` dans `pivot_table`.

**Erreurs possibles.** `shift(12)` sans `groupby` : on comparerait un produit au précédent dans le tableau. Données non triées. Oublier que les 12 premiers mois n'ont pas de YoY (NaN) : ils ne doivent pas être remplacés par 0.

## Correction — Exercice 12 : Panel de séries temporelles : tendances, alertes et autocorrélation

In [ ]:
large = air.pivot(index="date", columns="station", values="pm25")
print((large.isna().mean() * 100).round(1).nlargest(3).to_dict())

def plus_long_trou(s):
    na = s.isna()
    return na.groupby(na.ne(na.shift()).cumsum()).sum().max()
trous = large.apply(plus_long_trou)
print("stations avec un trou > 7 j :", trous[trous > 7].index.tolist())

# interpolation limitée aux petits trous
long_na = large.isna()
groupes = long_na.ne(long_na.shift()).cumsum()
lon = long_na.apply(lambda c: c.groupby(groupes[c.name]).transform("sum"))
petits = long_na & (lon <= 5)
comble = large.copy()
comble[petits] = large.interpolate(method="time")[petits]
print("NaN restants :", int(comble.isna().sum().sum()))

# pente de tendance pour toutes les stations (régression sur le temps, valeurs valides uniquement)
Y = comble.to_numpy()
t = np.arange(len(comble), dtype=float)[:, None]
valide = ~np.isnan(Y)
n = valide.sum(axis=0)
tm = np.where(valide, t, 0).sum(axis=0) / n
ym = np.nansum(Y, axis=0) / n
dt = np.where(valide, t - tm, 0.0)
dy = np.where(valide, np.nan_to_num(Y) - ym, 0.0)
pente_est = (dt * dy).sum(axis=0) / (dt ** 2).sum(axis=0)
res = np.where(valide, dy - pente_est * dt, 0.0)
se = np.sqrt((res ** 2).sum(axis=0) / (n - 2) / (dt ** 2).sum(axis=0))
tstat = pente_est / se
tend = pd.DataFrame({"pente": pente_est, "t": tstat}, index=comble.columns).sort_values("t", ascending=False)
print(tend.head(5).round(3))
print("stations avec t > 3 :", tend.index[tend["t"] > 3].tolist())

# z-score glissant et alertes mensuelles
rm = comble.rolling(30, min_periods=15).mean(); rs = comble.rolling(30, min_periods=15).std()
z = (comble - rm) / rs
print((z > 2).sum(axis=1).groupby(z.index.month).sum().to_dict())

# autocorrélation de ST-00 après retrait de tendance et saisonnalité
x = comble["ST-00"].interpolate().bfill().ffill().to_numpy()
tt = np.arange(len(x))
X = np.column_stack([np.ones_like(tt), tt, np.cos(2 * np.pi * tt / 366), np.sin(2 * np.pi * tt / 366)])
resid = x - X @ np.linalg.lstsq(X, x, rcond=None)[0]
acf = [np.corrcoef(resid[:-k], resid[k:])[0, 1] for k in range(1, 11)]
print(np.round(acf, 2))

# variables retardées sans fuite
a = air.sort_values(["station", "date"]).copy()
g = a.groupby("station")["pm25"]
a["lag1"] = g.shift(1); a["lag7"] = g.shift(7)
a["moy7_prec"] = g.transform(lambda s: s.shift(1).rolling(7, min_periods=4).mean())
print(a.head(10))

**Raisonnement.** Pour vectoriser une régression sur **40 colonnes à la fois**, on exprime pente et erreur-type avec des formules matricielles sur des tableaux `(366, 40)`, en masquant les valeurs absentes (`valide`). La pente d'une régression simple vaut `Σ(t−t̄)(y−ȳ) / Σ(t−t̄)²`. Les stations dégradées ressortent par leur statistique t élevée. L'autocorrélation d'un bruit AR(1) (coefficient 0,6) décroît géométriquement (≈ 0,6, 0,36, 0,22…) : on la voit **après** avoir retiré tendance et saison. Pour les variables retardées, `shift(1)` **avant** `rolling` garantit que le jour courant n'entre pas dans sa propre moyenne (sinon, fuite de données). Attention : les statistiques t de la tendance supposent des résidus indépendants ; or le bruit est autocorrélé (ρ ≈ 0,6), donc les erreurs-types sont **sous-estimées** et les t surestimés : c'est pourquoi quelques stations « normales » dépassent aussi t > 3. Les trois stations réellement dégradées restent très nettement en tête ; un test correct de tendance doit corriger l'autocorrélation (erreurs-types robustes, test de Mann-Kendall…).

**Erreurs possibles.** Calculer des autocorrélations sur la série brute : la saisonnalité annuelle crée des corrélations fortes à tous les décalages. Faire `rolling(7).mean().shift(1)` ou `shift(1).rolling(7)` : les deux sont équivalents ; l'erreur courante est de ne pas décaler du tout.

## Correction — Exercice 13 : Le paradoxe de Simpson (calculs de calcul rénal)

In [ ]:
effectifs = {("A", "petit"): (81, 87), ("A", "gros"): (192, 263), ("B", "petit"): (234, 270), ("B", "gros"): (55, 80)}
lignes = []
for (trait, taille), (s, n) in effectifs.items():
    lignes.append(pd.DataFrame({"traitement": trait, "taille": taille, "succes": np.r_[np.ones(s), np.zeros(n - s)].astype(int)}))
patients = pd.concat(lignes, ignore_index=True)

print((patients.groupby("traitement")["succes"].mean() * 100).round(1).to_dict())
print((patients.groupby(["taille", "traitement"])["succes"].mean().unstack() * 100).round(1))
print((pd.crosstab(patients["traitement"], patients["taille"], normalize="index") * 100).round(1))

def or_mh(d):
    num = den = 0.0
    for _, g in d.groupby("taille"):
        a = ((g["traitement"] == "A") & (g["succes"] == 1)).sum(); b = ((g["traitement"] == "A") & (g["succes"] == 0)).sum()
        c = ((g["traitement"] == "B") & (g["succes"] == 1)).sum(); dd = ((g["traitement"] == "B") & (g["succes"] == 0)).sum()
        n = len(g); num += a * dd / n; den += b * c / n
    return num / den

print("OR Mantel-Haenszel :", round(or_mh(patients), 2))

# bootstrap vectorisé : indices tirés par strate/traitement, comptes via binomiales
rng_b = np.random.default_rng(0)
cells = patients.groupby(["taille", "traitement"])["succes"].agg(["sum", "count"])
B = 2000
ors = np.zeros(B); num = np.zeros(B); den = np.zeros(B)
for taille in ["petit", "gros"]:
    sa, na = cells.loc[(taille, "A")]; sb, nb_ = cells.loc[(taille, "B")]
    a = rng_b.binomial(na, sa / na, B); c = rng_b.binomial(nb_, sb / nb_, B)
    b = na - a; d = nb_ - c; n_tot = na + nb_
    num += a * d / n_tot; den += b * c / n_tot
ors = num / den
print(np.percentile(ors, [2.5, 97.5]).round(2))

**Raisonnement.** Le traitement A est meilleur dans chaque strate (93 % vs 87 % et 73 % vs 69 %), mais B semble meilleur globalement (83 % vs 78 %) : c'est le **paradoxe de Simpson**. La cause : A est utilisé surtout sur les **gros calculs** (plus difficiles, taux de succès plus bas), B sur les petits. La taille du calcul est une variable de confusion. L'odds ratio de Mantel-Haenszel combine les strates en pondérant par leur taille et donne une estimation > 1 en faveur de A. Pour le bootstrap, comme les données sont binaires, on simule directement les comptes de succès de chaque cellule avec une loi binomiale (équivalent au rééchantillonnage des patients, mais sans construire des milliers de DataFrames). Le bootstrap donne un intervalle qui **inclut 1** (≈ [0,96 ; 2,3]) : malgré l'inversion de signe du paradoxe, l'avantage de A n'est pas établi à 95 % avec ces effectifs. Seule une **randomisation** (ou une analyse causale avec ajustement correct) permet de trancher.

**Erreurs possibles.** Conclure à partir des taux globaux. Agréger sans avoir regardé la répartition des strates. Appliquer MH sans vérifier que l'odds ratio est homogène entre strates.

## Correction — Exercice 14 : Test A/B : p-valeur, puissance et permutation vectorisée

In [ ]:
from scipy.stats import chi2_contingency, norm
nA, xA, nB, xB = 4800, 528, 5100, 627
pA, pB = xA / nA, xB / nB
p_pool = (xA + xB) / (nA + nB)
z = (pB - pA) / np.sqrt(p_pool * (1 - p_pool) * (1 / nA + 1 / nB))
p_z = 2 * (1 - norm.cdf(abs(z)))
chi = chi2_contingency([[xA, nA - xA], [xB, nB - xB]], correction=False)
print(round(pA, 4), round(pB, 4), round(z, 3), round(p_z, 5), round(chi[1], 5))

se = np.sqrt(pA * (1 - pA) / nA + pB * (1 - pB) / nB)
print("IC95% diff :", np.round([(pB - pA) - 1.96 * se, (pB - pA) + 1.96 * se], 4))

rng_a = np.random.default_rng(1)
N, K = nA + nB, xA + xB
xa_perm = rng_a.hypergeometric(K, N - K, nA, size=100_000)          # convertis tombant dans le "groupe A"
diff_perm = (K - xa_perm) / nB - xa_perm / nA
diff_obs = pB - pA
p_hyper = (np.abs(diff_perm) >= abs(diff_obs)).mean()
donnees = np.r_[np.ones(K), np.zeros(N - K)]
diffs = np.empty(10_000)
for i in range(10_000):
    rng_a.shuffle(donnees)
    diffs[i] = donnees[nA:].mean() - donnees[:nA].mean()
p_vrai = (np.abs(diffs) >= abs(diff_obs)).mean()
print("p permutation (hypergéométrique) :", p_hyper, "| (mélanges explicites) :", p_vrai)

# effet minimal détectable (formule) puis vérification par simulation
p0, n = 0.11, 4800
z_a, z_b = norm.ppf(0.975), norm.ppf(0.80)
delta = (z_a + z_b) * np.sqrt(2 * p0 * (1 - p0) / n)
print("MDE ≈", round(delta * 100, 2), "points")
sims = 2000
def puissance(effet):
    a = rng_a.binomial(n, p0, sims); b = rng_a.binomial(n, p0 + effet, sims)
    pp = (a + b) / (2 * n)
    zz = (b / n - a / n) / np.sqrt(pp * (1 - pp) * 2 / n)
    return (np.abs(zz) > 1.96).mean()
print("puissance simulée au MDE :", puissance(delta).round(3), "| pour +1,3 pt :", puissance(0.013).round(3))

# regards répétés sans effet réel
exps, regards, par_jour = 5000, 20, 250
conv_a = rng_a.binomial(par_jour, 0.11, (exps, regards)).cumsum(axis=1)
conv_b = rng_a.binomial(par_jour, 0.11, (exps, regards)).cumsum(axis=1)
nn = par_jour * np.arange(1, regards + 1)
pp = (conv_a + conv_b) / (2 * nn)
zz = (conv_b / nn - conv_a / nn) / np.sqrt(pp * (1 - pp) * 2 / nn)
print("faux positifs avec regards répétés :", (np.abs(zz) > 1.96).any(axis=1).mean().round(3))

**Raisonnement.** Le test z de deux proportions est équivalent au χ² sans correction (z² = χ²). Sous H₀, mélanger les étiquettes revient à tirer sans remise `nA` visiteurs parmi `N`, dont `K` convertis : le nombre de convertis dans A suit exactement une loi hypergéométrique, d'où un test de permutation à 100 000 tirages en une ligne. Pour la puissance, on simule des expériences et on compte la proportion de rejets de H₀ : au MDE elle vaut ≈ 80 %, mais pour l'effet observé (+1,3 point) elle n'est que d'environ 50 % : le test était **sous-dimensionné** pour cet effet, et un résultat à peine significatif a de bonnes chances de surestimer l'effet réel (« malédiction du gagnant »). Le dernier point illustre le **problème des regards multiples** (peeking) : s'arrêter dès que p < 0,05 fait passer le taux de faux positifs de 5 % à environ 20-25 %. Il faut fixer la taille d'échantillon à l'avance ou utiliser des méthodes séquentielles.

**Erreurs possibles.** Utiliser une variance non poolée pour le test d'hypothèse nulle (l'hypothèse nulle impose un taux commun). Interpréter p comme « probabilité que B soit meilleur » (c'est faux). Arrêter un test dès qu'il est significatif.

## Correction — Exercice 15 : Construire un pipeline de nettoyage testable et traçable

In [ ]:
import re, functools, unicodedata
journal = []

def tracer(f):
    @functools.wraps(f)
    def wrapper(df, *a, **k):
        t = time.perf_counter(); n0 = len(df)
        out = f(df, *a, **k)
        journal.append({"etape": f.__name__, "avant": n0, "apres": len(out), "ms": round((time.perf_counter() - t) * 1000, 1)})
        return out
    return wrapper

PAYS = {"benin": "Bénin", "togo": "Togo", "cote d'ivoire": "Côte d'Ivoire", "ghana": "Ghana", "nigeria": "Nigeria"}

def _sans_accent(s):
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

@tracer
def dedoublonner(df): return df.drop_duplicates(subset="id", keep="first").copy()

@tracer
def normaliser_pays(df):
    d = df.copy()
    cle = d["pays"].astype("string").str.strip().str.lower().map(lambda s: _sans_accent(s) if isinstance(s, str) else s)
    d["pays"] = cle.map(PAYS)
    return d

@tracer
def parser_dates(df):
    d = df.copy()
    iso = pd.to_datetime(d["date"], format="%Y-%m-%d", errors="coerce")
    fr = pd.to_datetime(d["date"], format="%d/%m/%Y", errors="coerce")
    d["date"] = iso.fillna(fr)
    return d

REGEX_EMAIL = r"^[\w.+-]+@[\w-]+\.[\w.-]+$"
@tracer
def valider_emails(df):
    d = df.copy(); d["email_valide"] = d["email"].astype("string").str.match(REGEX_EMAIL).fillna(False).astype(bool)
    return d

@tracer
def nettoyer_age(df):
    d = df.copy(); d["age"] = pd.to_numeric(d["age"], errors="coerce").where(lambda s: s.between(16, 100)); return d

@tracer
def convertir_montant(df):
    d = df.copy(); d["montant"] = pd.to_numeric(d["montant"], errors="coerce"); return d

@tracer
def ajouter_features(df):
    d = df.copy()
    d["mois"] = d["date"].dt.month; d["jour_sem"] = d["date"].dt.dayofweek
    d["domaine"] = d["email"].where(d["email_valide"]).str.split("@").str[1]
    return d

def valider(df):
    assert df["id"].is_unique, "id non uniques"
    assert df["pays"].isin(PAYS.values()).all(), "pays non canoniques"
    assert df["date"].notna().all(), "dates non parsées"
    assert df["age"].dropna().between(16, 100).all(), "âge hors plage"
    assert df["montant"].dropna().ge(0).all(), "montant négatif"
    return df

def pipeline(df):
    return (df.pipe(dedoublonner).pipe(normaliser_pays).pipe(parser_dates).pipe(valider_emails)
              .pipe(nettoyer_age).pipe(convertir_montant).pipe(ajouter_features).pipe(valider))

avant = inscriptions.copy()
propre1 = pipeline(inscriptions)
print("entrée inchangée :", inscriptions.equals(avant))
journal_1 = list(journal); journal.clear()
propre2 = pipeline(propre1.drop(columns=["email_valide", "mois", "jour_sem", "domaine"]).assign(date=lambda d: d["date"].dt.strftime("%Y-%m-%d")))
print("idempotent :", propre1.reset_index(drop=True).equals(propre2.reset_index(drop=True)))

# tests unitaires
mini = pd.DataFrame({"id": [1, 2, 3], "pays": [" BENIN ", "Cote d'Ivoire", "Atlantide"]})
res = normaliser_pays(mini)
assert res["pays"].tolist()[:2] == ["Bénin", "Côte d'Ivoire"] and pd.isna(res["pays"].iloc[2])
mail = valider_emails(pd.DataFrame({"email": ["a@b.com", "pas-un-email", "x.y+z@dom.fr", None]}))
assert mail["email_valide"].tolist() == [True, False, True, False]
print(pd.DataFrame(journal_1))
print((propre1.isna().mean() * 100).round(2).to_dict())
print((~propre1["email_valide"]).sum(), propre1["pays"].value_counts().to_dict())

**Raisonnement.** Un pipeline fiable repose sur des **fonctions pures** (pas d'effet de bord sur l'entrée : `copy()` en début de chaque étape), **composables** (`pipe`), **traçables** (le décorateur journalise lignes et durée) et **vérifiables** (`valider`, tests unitaires). L'idempotence (rejouer le pipeline sur sa sortie ne change rien) est une propriété essentielle pour pouvoir relancer un traitement sans crainte. Les formats de date sont gérés en essayant chaque format avec `errors="coerce"` puis en comblant par `fillna` (plus sûr que `format="mixed"` quand on connaît les formats attendus). `str.match` avec une regex valide les e-mails.

**Erreurs possibles.** Modifier le DataFrame d'entrée dans une étape (effet de bord) : on perd la reproductibilité. Utiliser `inplace=True` dans un pipeline. Valider avec des `print` au lieu d'`assert` : l'erreur passe inaperçue en production.

## Correction — Exercice 16 : ACP à la main : valeurs propres, SVD et reconstruction

In [ ]:
rng_pca = np.random.default_rng(900)
facteurs = rng_pca.normal(size=(500, 3)) * np.array([3.0, 2.0, 1.2])
charges = rng_pca.normal(size=(3, 8))
X = facteurs @ charges + rng_pca.normal(0, 0.5, (500, 8)) + np.arange(8) * 5

Z = (X - X.mean(axis=0)) / X.std(axis=0, ddof=1)
C = np.cov(Z, rowvar=False)
vals, vecs = np.linalg.eigh(C)
ordre = np.argsort(vals)[::-1]; vals, vecs = vals[ordre], vecs[:, ordre]
part = vals / vals.sum(); cum = part.cumsum()
print(part.round(3), cum.round(3))
print("k pour 90 % :", int(np.searchsorted(cum, 0.90) + 1))

scores = Z @ vecs                                               # (500, 8) toutes les composantes
mse_k = np.array([((Z - scores[:, :k] @ vecs[:, :k].T) ** 2).mean() for k in range(1, 9)])
theorie = np.array([vals[k:].sum() / 8 * (500 - 1) / 500 for k in range(1, 9)])
print(mse_k.round(4), np.allclose(mse_k, theorie))

U, s, Vt = np.linalg.svd(Z, full_matrices=False)
print(np.allclose(s ** 2 / (len(Z) - 1), vals))
print(np.allclose(np.abs(Vt), np.abs(vecs.T)))

try:
    from sklearn.decomposition import PCA
    p = PCA().fit(Z)
    print("sklearn concordant :", np.allclose(p.explained_variance_, vals), np.allclose(np.abs(p.components_), np.abs(vecs.T)))
except ImportError:
    print("scikit-learn non installé")

print(np.round(vecs[:, 0], 2))
Xc = X - X.mean(axis=0)                                       # centrage seul, sans réduction
v_nr, w_nr = np.linalg.eigh(np.cov(Xc, rowvar=False))
print("sans réduction : part de la 1ère composante", (v_nr[-1] / v_nr.sum()).round(3), "| charges", np.round(np.abs(w_nr[:, -1]), 2))
print("écarts-types des variables :", X.std(axis=0).round(1))

**Raisonnement.** L'ACP diagonalise la matrice de covariance des données centrées-réduites : les vecteurs propres sont les axes de variance maximale et les valeurs propres les variances le long de ces axes. L'erreur de reconstruction en gardant `k` composantes vaut la somme des variances des composantes **écartées** (à une normalisation `n/(n−1)` près, d'où le facteur dans la comparaison). La SVD de `Z` donne `Z = U S Vᵗ` : les lignes de `Vᵗ` sont les composantes et `s²/(n−1)` les valeurs propres — c'est ainsi que les bibliothèques calculent l'ACP en pratique (plus stable numériquement). Seuls 3 facteurs latents génèrent la structure : les 3 premières composantes capturent l'essentiel de la variance.

**Erreurs possibles.** Oublier de réduire (diviser par l'écart-type) : la variable de plus grande échelle domine la 1ʳᵉ composante. Comparer les vecteurs propres sans tenir compte de l'indétermination de signe.

## Correction — Exercice 17 : Fenêtres glissantes vectorisées : corrélation, drawdown et recherche de motif

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view as swv

# 1. corrélation glissante
rd1 = np.diff(prix1); rd2 = np.diff(prix2)
W = 250
A = swv(rd1, W); B = swv(rd2, W)
Am = A - A.mean(axis=1, keepdims=True); Bm = B - B.mean(axis=1, keepdims=True)
corr_g = (Am * Bm).sum(axis=1) / np.sqrt((Am ** 2).sum(axis=1) * (Bm ** 2).sum(axis=1))
for i in (0, 2000, 4000):
    assert np.isclose(corr_g[i], np.corrcoef(rd1[i:i + W], rd2[i:i + W])[0, 1])
bascule = int(np.argmax(corr_g > 0.5))
print(len(corr_g), "bascule vers la fenêtre", bascule, "(fin de fenêtre à t =", bascule + W, ")", corr_g[[0, -1]].round(2))

# 2. drawdown maximal glissant
F = swv(prix1, 500)
pic = np.maximum.accumulate(F, axis=1)
dd = ((F - pic) / pic).min(axis=1)
print(F.shape, round(dd.min() * 100, 1), "% pire drawdown glissant")

# 3. recherche de motif par z-score de fenêtres
m = len(motif)
F2 = swv(prix_motif, m)
Fz = (F2 - F2.mean(axis=1, keepdims=True)) / F2.std(axis=1, keepdims=True)
mz = (motif - motif.mean()) / motif.std()
dist = np.sqrt(((Fz - mz) ** 2).sum(axis=1))
trouves = []
d = dist.copy()
for _ in range(3):
    i = int(d.argmin()); trouves.append(i); d[max(0, i - m):i + m] = np.inf       # écarte les chevauchements
print(sorted(trouves))

# 4. mémoire
print(F2.nbytes, F2.base is not None, np.shares_memory(F2, prix_motif), F2.flags.writeable)

**Raisonnement.** `sliding_window_view` crée une **vue** `(n_fenêtres, taille)` qui ne copie rien (les fenêtres se chevauchent en mémoire) : on peut alors calculer moyennes, écarts-types ou corrélations de *toutes* les fenêtres par des opérations vectorisées sur l'axe 1. `nbytes` renvoie la taille « logique » (nombre de fenêtres × taille × 8), mais la mémoire réellement occupée reste celle de la série d'origine tant qu'on ne matérialise pas de résultat intermédiaire. Pour le drawdown, `np.maximum.accumulate(axis=1)` donne le plus haut atteint jusque-là dans chaque fenêtre. La z-normalisation rend la recherche de motif insensible au niveau et à l'amplitude. Pour une corrélation glissante, centrer chaque fenêtre sur sa propre moyenne est indispensable.

**Erreurs possibles.** Modifier la vue (la vue est en lecture seule par défaut, sinon on corromprait la série). Calculer `np.corrcoef` dans une boucle de 4 700 fenêtres (lent). Chercher le motif sans normaliser : le niveau du prix domine la distance.

## Correction — Exercice 18 : PROJET 1 — Moto-taxis de Cotonou : analyse complète

In [ ]:
# ---------------- Étape 1 : qualité ----------------
n0 = len(courses)
print("doublons exacts :", int(courses.duplicated().sum()), "| id_course dupliqués :", int(courses["id_course"].duplicated().sum()))
c = courses.drop_duplicates(subset="id_course", keep="first").copy()      # id_course est la clé primaire
print("lignes supprimées :", n0 - len(c))
c["duree_min"] = c["duree_min"].abs()                                 # signe inversé par erreur : on récupère la valeur absolue
terminee = c["statut"] == "Terminée"
vitesse = c["distance_km"] / (c["duree_min"] / 60)
aberrant = terminee & ((c["distance_km"] > 30) | (vitesse > 60))
print("distances aberrantes :", int(aberrant.sum()))
c.loc[aberrant, "distance_km"] = c.loc[aberrant, "distance_km"] / 10  # erreur d'un facteur 10
c["jour"] = c["ts"].dt.normalize(); c["heure"] = c["ts"].dt.hour; c["jds"] = c["ts"].dt.dayofweek
t = c[c["statut"] == "Terminée"].copy()
assert (c.loc[~terminee, "prix_fcfa"] == 0).all() and (t["duree_min"] > 0).all() and t["distance_km"].between(0.5, 30).all()
assert not c["id_course"].duplicated().any() and ((t["distance_km"] / (t["duree_min"] / 60)) < 60).all()

# ---------------- Étape 2 : demande ----------------
c = c.merge(meteo, left_on="jour", right_on="date", how="left").drop(columns="date")
c["pluie"] = c["pluie_mm"] > 5
nb_jours_jds = c.groupby("jds")["jour"].nunique()
tab = c.groupby(["heure", "jds"]).size().unstack() / nb_jours_jds
print("heures de pointe (moyenne sur la semaine) :", tab.mean(axis=1).nlargest(4).round(0).to_dict())
quot = c.groupby("jour").size().rename("n").to_frame().join(meteo.set_index("date")["pluie_mm"])
quot["pluie"] = quot["pluie_mm"] > 5; quot["jds"] = quot.index.dayofweek
eff = quot.groupby(["jds", "pluie"])["n"].mean().unstack()
eff["effet_%"] = (eff[True] / eff[False] - 1) * 100
print(eff.round(1)); print("effet pluie moyen :", round(eff["effet_%"].mean(), 1), "%")

# ---------------- Étape 3 : tensions ----------------
c["pointe"] = np.where(c["heure"].isin([7, 8, 9, 17, 18, 19]), "pointe", "creux")
c["annulee"] = c["statut"] != "Terminée"
tens = c.groupby(["zone_depart", "pointe"]).agg(n=("annulee", "size"), taux=("annulee", "mean")).unstack()
tens_t = (c[c["annulee"] == False].groupby(["zone_depart", "pointe"])["duree_min"].mean().unstack())
print((tens["taux"] * 100).round(1).sort_values("pointe", ascending=False).head(4))
from scipy.stats import norm
pire = c.groupby("zone_depart")["annulee"].agg(["sum", "count"]); pire["taux"] = pire["sum"] / pire["count"]
z_pire = pire["taux"].idxmax(); p0 = c["annulee"].mean()
z = (pire.loc[z_pire, "taux"] - p0) / np.sqrt(p0 * (1 - p0) / pire.loc[z_pire, "count"])
print(z_pire, round(pire.loc[z_pire, "taux"] * 100, 1), "% vs", round(p0 * 100, 1), "% | z =", round(z, 1), "p =", 2 * (1 - norm.cdf(abs(z))))

# ---------------- Étape 4 : conducteurs ----------------
cond = t.groupby("id_conducteur").agg(courses=("id_course", "size"), revenu=("prix_fcfa", "sum"), minutes=("duree_min", "sum"))
cond["rev_course"] = cond["revenu"] / cond["courses"]; cond["rev_heure"] = cond["revenu"] / (cond["minutes"] / 60)
def gini(x):
    x = np.sort(np.asarray(x, dtype=float)); n = len(x)
    return (2 * np.arange(1, n + 1) - n - 1) @ x / (n * x.sum())
top10 = cond["revenu"].nlargest(len(cond) // 10).sum() / cond["revenu"].sum()
print("Gini :", round(gini(cond["revenu"]), 3), "| part du top 10 % :", round(top10 * 100, 1), "%")
att = c.groupby("id_conducteur").agg(n=("id_course", "size"), ann_c=("statut", lambda s: (s == "Annulée conducteur").sum()))
att = att[att["n"] >= 100]; p_glob = att["ann_c"].sum() / att["n"].sum()
att["z"] = (att["ann_c"] / att["n"] - p_glob) / np.sqrt(p_glob * (1 - p_glob) / att["n"])
print("conducteurs suspects (z > 3) :", int((att["z"] > 3).sum()), "sur", len(att), "| attendus par hasard ≈", round(len(att) * (1 - norm.cdf(3)), 1))

# ---------------- Étape 5 : tarification ----------------
t = t.merge(meteo, left_on="jour", right_on="date", how="left").drop(columns="date"); t["pluie"] = t["pluie_mm"] > 5
X = np.column_stack([np.ones(len(t)), t["distance_km"], t["pluie"].astype(float), t["heure"].isin([7, 8, 9, 17, 18, 19]).astype(float)])
y = t["prix_fcfa"].to_numpy()
beta, *_ = np.linalg.lstsq(X, y, rcond=None)
res = y - X @ beta
print(dict(zip(["base", "FCFA/km", "pluie", "pointe"], beta.round(1))), "| R² =", round(1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum(), 4))
print("résidus : moyenne", round(res.mean(), 2), "écart-type", round(res.std(), 1))

# ---------------- Étape 6 : prévision ----------------
h = c.set_index("ts").resample("h").size()
h = h.reindex(pd.date_range("2024-01-01", "2024-12-31 23:00", freq="h"), fill_value=0)
test = h["2024-12-04":]; train = h[:"2024-12-03 23:00"]
pred_a = h.shift(24 * 7).loc[test.index]
prof = train.groupby([train.index.dayofweek, train.index.hour]).mean()
pred_b = pd.Series(prof.reindex(list(zip(test.index.dayofweek, test.index.hour))).to_numpy(), index=test.index)
for nom, p in [("semaine précédente", pred_a), ("profil moyen", pred_b)]:
    mae = (test - p).abs().mean(); mask = test >= 5
    mape = ((test - p).abs() / test)[mask].mean() * 100
    print(f"{nom:<20} MAE = {mae:.2f} | MAPE (heures >= 5 courses) = {mape:.1f} %")

# ---------------- Étape 7 : chiffrage ----------------
perte = c[c["annulee"]]["zone_depart"].value_counts().head(3)
print("courses annulées (top zones) :", perte.to_dict(), "| manque à gagner estimé :", int(perte.sum() * t["prix_fcfa"].mean()), "FCFA")

**Raisonnement.** **Démarche.** (1) Dédoublonner sur la **clé primaire** `id_course` plutôt que sur des lignes entières : une partie des doublons n'est pas exacte (l'un des deux exemplaires a subi une autre corruption), `drop_duplicates()` seul en aurait laissé quelques-uns ; les durées négatives sont des erreurs de signe (récupérables) ; les distances aberrantes ont un facteur 10 évident (vitesses > 60 km/h) ; les annulations ont prix 0 et NaN pour la distance : cohérent. (2) L'effet de la pluie se mesure **à jour de semaine égal** pour ne pas confondre avec la saisonnalité hebdomadaire (les jours de pluie ne tombent pas uniformément dans la semaine). On trouve environ +20 % de demandes, valeur injectée dans les données. (3) Les annulations dépendent de la pluie, de l'heure de pointe et de la zone (Dantokpa, marché) ; un test sur proportion (z de la zone vs taux global) en confirme la significativité. (4) Les revenus sont inégalement répartis car l'activité est log-normale (Gini ≈ 0,25 ; les 10 % de conducteurs les plus actifs gagnent ≈ 20 % du total). Pour les annulations, quelques conducteurs dépassent z > 3, mais **à peine plus que le nombre attendu par hasard** (≈ 0,8 sur 598) : il n'y a pas de comportement anormal démontré. Il faut toujours comparer le nombre de « suspects » au nombre attendu sous H₀ (tests multiples) avant d'accuser quelqu'un. (5) Les coefficients retrouvent la règle tarifaire (≈ 140 FCFA/km ; majoration pluie ≈ 15 % du prix moyen, soit ≈ 190 FCFA ; la pointe n'a pas d'effet, coefficient ≈ 0). La base estimée (≈ 273) est un peu inférieure à 300 car la majoration est **multiplicative** (interaction distance × pluie) alors que le modèle additif la répartit sur les autres coefficients ; ajouter le terme `distance × pluie` la ferait coïncider. (6) Le profil moyen lisse mieux le bruit que la « semaine précédente », qui hérite du bruit d'une seule semaine ; le MAPE est calculé hors heures creuses pour éviter les divisions quasi nulles. (7) Chaque recommandation doit citer un chiffre issu des étapes précédentes et une limite (corrélation ≠ causalité, une seule année, etc.).

**Erreurs possibles.** Supprimer les « doublons » sur `id_course` seul sans vérifier que les lignes sont identiques. Calculer l'effet pluie sans contrôler le jour de la semaine. Interpréter un coefficient de régression sans regarder les unités. Calculer un MAPE avec des valeurs réelles nulles. Oublier de comparer le nombre de conducteurs « suspects » au nombre attendu par hasard (tests multiples).

## Correction — Exercice 19 : PROJET 2 — Rendements agricoles : analyse complète

In [ ]:
# -------------------- Étape 1 --------------------
d = (campagnes.merge(fermes, on="id_ferme", how="left", validate="many_to_one")
              .merge(pluviometrie, on=["region", "annee"], how="left", validate="many_to_one"))
assert len(d) == 3600 and not d.duplicated(["id_ferme", "annee"]).any()
print("engrais manquants :", int(d["engrais_kg_ha"].isna().sum()), "| rendements manquants :", int(d["rendement_t_ha"].isna().sum()))
med = d.groupby("culture")["rendement_t_ha"].transform("median")
q = d.groupby("culture")["rendement_t_ha"].transform(lambda s: s.quantile(0.75) - s.quantile(0.25))
erreur_unite = d["rendement_t_ha"] > med + 3 * q.clip(lower=0.3) * 2
d.loc[erreur_unite, "rendement_t_ha"] = d.loc[erreur_unite, "rendement_t_ha"] / 10
print("erreurs d'unité corrigées :", int(erreur_unite.sum()))
d = d.dropna(subset=["rendement_t_ha"]).copy()
d["grande"] = d["taille_ha"] > 5
d["engrais_kg_ha"] = d["engrais_kg_ha"].fillna(d.groupby(["culture", "grande"])["engrais_kg_ha"].transform("median"))
assert d["engrais_kg_ha"].notna().all()

# -------------------- Étape 2 --------------------
print(d.groupby("culture")["rendement_t_ha"].mean().round(2).to_dict(), d.groupby("semences")["rendement_t_ha"].mean().round(2).to_dict())
print(d.groupby("irrigation")["rendement_t_ha"].mean().round(2).to_dict(), d.groupby("annee")["rendement_t_ha"].mean().round(2).to_dict())
mais = d[d["culture"] == "Maïs"]
print(mais.groupby(pd.cut(mais["pluie_mm"], 6), observed=True)["rendement_t_ha"].mean().round(2))
print("corr(taille, adoption) :", round(np.corrcoef(d["taille_ha"], (d["semences"] == "améliorées").astype(float))[0, 1], 2))

# -------------------- Étape 3 : OLS --------------------
def design(df):
    return np.column_stack([
        np.ones(len(df)), (df["culture"] == "Soja"), (df["culture"] == "Coton"),
        (df["semences"] == "améliorées"), df["irrigation"].astype(float),
        df["engrais_kg_ha"] / 100, (df["engrais_kg_ha"] / 100) ** 2,
        (df["pluie_mm"] - 1000) / 100, ((df["pluie_mm"] - 1000) / 100) ** 2,
        (df["sol"] == "limoneux"), (df["sol"] == "argileux"), np.log(df["taille_ha"])]).astype(float)
noms = ["const", "Soja", "Coton", "semences_amél", "irrigation", "engrais/100", "engrais²", "pluie_c", "pluie_c²", "limoneux", "argileux", "log_taille"]
X = design(d); y = np.log(d["rendement_t_ha"].to_numpy())
beta = np.linalg.lstsq(X, y, rcond=None)[0]
res = y - X @ beta
n, p = X.shape
sigma2 = (res ** 2).sum() / (n - p)
se = np.sqrt(np.diag(sigma2 * np.linalg.inv(X.T @ X)))
r2 = 1 - (res ** 2).sum() / ((y - y.mean()) ** 2).sum()
print(pd.DataFrame({"coef": beta, "se": se, "t": beta / se}, index=noms).round(3)); print("R² =", round(r2, 3), "| RMSE =", round(np.sqrt((res ** 2).mean()), 3))
print("effet semences améliorées :", round((np.exp(beta[3]) - 1) * 100, 1), "% | optimum de pluie ≈", round(1000 - 100 * beta[7] / (2 * beta[8]) * 1, 0), "mm")

# -------------------- Étape 4 : validation temporelle --------------------
tr = d["annee"] <= 2022
b_tr = np.linalg.lstsq(X[tr], y[tr], rcond=None)[0]
rmse_mod = np.sqrt(((y[~tr] - X[~tr] @ b_tr) ** 2).mean())
base = d[tr].assign(ly=y[tr]).groupby("culture")["ly"].mean()
rmse_base = np.sqrt(((y[~tr] - d.loc[~tr, "culture"].map(base).to_numpy()) ** 2).mean())
print("RMSE test : modèle", round(rmse_mod, 3), "| baseline", round(rmse_base, 3))

# -------------------- Étape 5 : efficacité --------------------
d["res"] = res
eff = d.groupby("id_ferme")["res"].mean()
e1 = d[d["annee"] <= 2021].groupby("id_ferme")["res"].mean(); e2 = d[d["annee"] >= 2022].groupby("id_ferme")["res"].mean()
print("persistance :", round(e1.corr(e2), 2), "| corrélation avec la compétence cachée :", round(eff.corr(secret), 2))
top = eff.nlargest(int(len(eff) * 0.1)).index
print(fermes.set_index("id_ferme").loc[top, "region"].value_counts(normalize=True).round(2).to_dict())

# -------------------- Étape 6 : impact des semences --------------------
brut = d.groupby("semences")["rendement_t_ha"].mean(); print("écart brut :", round((brut["améliorées"] / brut["traditionnelles"] - 1) * 100, 1), "%")
rng_b = np.random.default_rng(0)
ids = d["id_ferme"].to_numpy(); uniq = np.unique(ids)
groupes = {f: np.where(ids == f)[0] for f in uniq}
coefs = []
for _ in range(300):
    tir = rng_b.choice(uniq, len(uniq), replace=True)
    idx = np.concatenate([groupes[f] for f in tir])
    coefs.append(np.linalg.lstsq(X[idx], y[idx], rcond=None)[0][3])
lo, hi = np.percentile((np.exp(coefs) - 1) * 100, [2.5, 97.5])
print(f"effet ajusté : {(np.exp(beta[3]) - 1) * 100:.1f} % [IC95 bootstrap : {lo:.1f} ; {hi:.1f}] (vrai effet simulé : +15 %)")

# -------------------- Étape 7 : contrefactuel --------------------
d24 = d[(d["annee"] == 2024) & (d["semences"] == "traditionnelles")].copy()
fact = d24.assign(semences="améliorées")
gain = np.exp(design(fact) @ beta) - np.exp(design(d24) @ beta)
d24["gain_t_ha"] = gain; d24["gain_t"] = gain * d24["taille_ha"]
print(d24.groupby("region")["gain_t"].sum().round(0).to_dict(), "| total :", round(d24["gain_t"].sum(), 0), "tonnes")

**Raisonnement.** **Démarche.** (1) Les rendements ×10 sont des erreurs d'unité visibles par rapport à la distribution de leur culture ; les corriger (÷10) est préférable à les supprimer car l'information existe. L'imputation de l'engrais se fait par groupe, **sans** les valeurs fautives. (2) La relation rendement/pluie est en cloche (optimum ≈ 1 000 mm) ; les grandes fermes adoptent davantage les semences améliorées (**confusion**). (3) On modélise `log(rendement)` : les coefficients s'interprètent alors en pourcentages (`exp(β) − 1`), et l'effet de la pluie est quadratique (l'optimum se trouve à `−β₁/(2β₂)`). Les erreurs-types viennent de `σ² (XᵀX)⁻¹`. (4) La validation temporelle est plus honnête qu'un découpage aléatoire (on prévoit le futur) ; le modèle bat nettement la baseline par culture. (5) Le résidu moyen par ferme estime sa « compétence » ; sa persistance entre périodes et sa corrélation avec la compétence cachée valident la méthode (en pratique on n'a pas ce contrôle). (6) L'écart brut surestime l'effet réel car les fermes qui adoptent les semences sont aussi plus grandes, mieux dotées en engrais et en irrigation : en ajustant sur ces variables, on retrouve l'effet de +15 % injecté ; le **bootstrap par ferme** (rééchantillonnage de grappes) respecte la corrélation entre campagnes d'une même ferme. (7) Le contrefactuel multiplie le gain prédit par la surface ; c'est une estimation conditionnelle au modèle, qui suppose que l'effet est identique pour toutes les fermes.

**Erreurs possibles.** Rééchantillonner des lignes au lieu de fermes dans le bootstrap (sous-estime l'incertitude, les campagnes d'une même ferme étant corrélées). Interpréter l'écart brut comme un effet causal. Découper aléatoirement train/test dans un panel temporel (fuite d'information). Oublier de passer au `log` : les erreurs ne sont plus homoscédastiques et les coefficients ne sont pas des pourcentages.